In [1]:
#@title Fill in these options first!

#@markdown You should leave all of these boxes checked and set `year` to the year you're analyzing. You only need to worry about changing this if the Colorado or New Mexico spills websites aren't working.

#year = input("What year do you want to analyze?\n")

import ipywidgets as widgets

#@markdown What year do you want to analyze?
year = 2025 #@param {type:"integer"}
lastyear = year - 1
#@markdown Use Google Drive to store data?
usedrive = True #@param {type:"boolean"}
rootdir = '/content/drive/Shareddrives/Long Term Projects/Western Oil & Gas Spills Tracker'
#@markdown Download Colorado spills data from the COGCC website?
codownload = True #@param {type:"boolean"}
#@markdown Download New Mexico spills data from the OCD website?
nmdownload = True #@param {type:"boolean"}
#@markdown ---

print('Done!')


Done!


In [2]:
#@title Backup filenames
#@markdown These are only used if you're uploading your own data files for Colorado and New Mexico. If Colorado and New Mexico's websites are working, the data files will automatically download, in which case you don't need to change these filenames at all.

#@markdown If you need to use your own files, put them in the current year's folder in the [shared Google drive](https://drive.google.com/drive/u/0/folders/1wvmk2qAl5dEZvmvAwQIbx4FmTFzds5oh).

#@markdown For Wyoming, email patrick.amole1@wyo.gov and ask for data (send last year’s file as example).

#@markdown Name those spreadsheets `WyomingSpills[20xx].xls` and put them in [your folder for this year's spills report](https://drive.google.com/drive/u/0/folders/1wvmk2qAl5dEZvmvAwQIbx4FmTFzds5oh). Put last year's spreadsheet in the same folder as well. If this notebook gives you an error reading an Excel sheet when you run the Wyoming section, export the sheets to CSV and use that instead.

#@markdown **When you've got both Wyoming files in place, run this cell to set the filenames.**

#@markdown Colorado spills filename:
cofilename = 'ColoradoSpills.xlsx' #@param {type:'string'}
#@markdown New Mexico spills filename:
nmfilename = 'NewMexicoSpills.htm' #@param {type:'string'}
#@markdown Wyoming spills filenames:
wyfilename1 = 'WyomingSpills2025.xlsx' #@param {type:'string'}
wyfilename2 = 'WyomingSpills2024.xlsx' #@param {type:'string'}

print('Done!')

Done!


# Setup

Run these cells all together. You don't need to change anything. This section connects to Google Drive and creates folders for exporting the spills data. It sets a handful of helper functions that are used across all three states.

In [3]:
#@title Import libraries

# (Google Colab has all of these. You may need to use pip if you're running this
# on your own)
import pandas as pd
import numpy as np
import os
import csv
import re
import math
import urllib.request as request
import zipfile
import pdb

# Suppress warnings about future versions
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [4]:
#@title Connect to Google Drive, create a folder if necessary
if usedrive:
  print("Connecting to Google Drive...")
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    rootdir = '/content/drive/Shareddrives/Long Term Projects/Spills Tracker'
    workingdir = f'{rootdir}/{str(year)}'
    print(f"Working directory: {workingdir}")
    if not os.path.exists(workingdir):
      print(f"Creating directory: {workingdir} ...")
      os.mkdir(workingdir)
    else:
      print(f"Directory already exists.")

    # make a link to the path with our data for this year. it's /content/data.

    # remove the link if it already exists (in case you run multiple years)
    if os.path.islink('/content/data'):
      print("Link to /content/data already exists, removing...")
      os.unlink('/content/data')
      print("Link removed.")

    # now create the link
    try:
      print("Creating symlink to /content/data...")
      os.symlink(workingdir, '/content/data')
      print("Link created.")
    except:
      print("Something went wrong creating the link.")

  except:
    if not os.path.exists('/content/data'):
      os.mkdir('/content/data')
    print("Google Drive not mounted. Using /content/data as working directory.")

if not usedrive:
  print("Not using Google Drive.")

  # Unlink the data folder
  if os.path.islink('/content/data'):
    print("Unlinking /content/data ...")
    os.unlink('/content/data')

  try:
    print("Disconnecting Google Drive...")
    drive.flush_and_unmount()
    print("Disconnected.")
  except:
    print('Disconnecting failed.')

  if not os.path.exists('/content/data'):
    print("Creating /content/data folder.")
    os.mkdir('/content/data')

#now set /content/data as our working directory from here on out.
workingdir = '/content/data'

Connecting to Google Drive...
Mounted at /content/drive
Working directory: /content/drive/Shareddrives/Long Term Projects/Spills Tracker/2025
Directory already exists.
Creating symlink to /content/data...
Link created.


In [5]:
#@title Create a folder structure for exports
workingdir = '/content/data'

# Does our data folder exist?
if not os.path.exists(workingdir):
  print(f'Creating {workingdir} directory.')
  os.mkdir(workingdir)

folder_names = ["colorado",
                "newmexico",
                "wyoming"]

for folder_name in folder_names:
  try:
    os.mkdir(f"{workingdir}/{folder_name}")
  except FileExistsError:
    print(f"Folder {workingdir}/{folder_name} already exists.")
  else:
    print(f"Folder {workingdir}/{folder_name} created successfully.")

# Line separator for text files
hr = "---------------\n"

# set the display format for floating point numbers
pd.options.display.float_format = '{:,.0f}'.format

Folder /content/data/colorado created successfully.
Folder /content/data/newmexico created successfully.
Folder /content/data/wyoming created successfully.


In [6]:
#@title Helper functions across all states (unit conversions, etc)

# For converting barrels to gallons
def gallons(x):
  return x * 42

# round to a per day average
def perday(num):
  return num/365

# Convert mcfs to cubic feet
def cubic(num):
  return num * 1000

# Pretty print large numbers
comma = "{:,.0f}"

# Pretty print small numbers
small = "{:,.1f}"

# Colorado
The first time you run the spills report, expand the `Import and setup` section.

## Import and setup
Expand the `Filter by date, clean up operator names` section and edit the cell titled `Clean up operators using similar names`. There are instructions in that cell.

You may need to run this section a few times to make sure the list of operators is clean and doesn't contain duplicates that need to be consolidated into one operator name.

In [7]:
#@title Download the latest Colorado spills data if the box is checked
if codownload:
  try:
    url = 'https://cogcc.state.co.us/documents/data/downloads/environmental/Spills.zip'
    zipfilename = (f"{workingdir}/Spills.zip")
    extractfile = ("Spills.xlsx")
    # Download the zip file
    print(f"Downloading {zipfilename}...")
    request.urlretrieve(url, zipfilename)
    # Extract the zip file
    print(f"Extracting {extractfile}...")
    with zipfile.ZipFile(zipfilename, 'r') as zip_ref:
      zip_ref.extract(extractfile, path=workingdir)

    # Rename it -- cofilename is set at the very top of this notebook
    os.rename(f"{workingdir}/{extractfile}", f"{workingdir}/{cofilename}")

    # Did this work?
    if os.path.exists(f"{workingdir}/{cofilename}"):
      print(f"{cofilename} successfully downloaded.")
    else:
      print("Something went wrong, try downloading manually.")

  except:
    print("Something went wrong, try downloading manually.")


Extracting Spills.xlsx...
ColoradoSpills.xlsx successfully downloaded.


In [8]:
#@title Import the Excel file
# Upload it to the /data folder in the notebook first,
# or put a link on Google Drive. The dataframe for all the spills is
# called "spills".

file = (f"{workingdir}/{cofilename}")

spills = pd.read_excel(file, engine='openpyxl')

#which column has our unique identifier? We'll use this throughout.
id = "Tracking #"

### Filter by date, clean up operator names
You'll need to check and edit the second cell in this section!

In [9]:
# force date columns to date datatype
spills['Initial Report Date'] = pd.to_datetime(spills['Initial Report Date'])
spills['Date of Discovery'] = pd.to_datetime(spills['Date of Discovery'])

#extract the month and year of those report dates into their own columns
spills['Month'] = spills['Date of Discovery'].dt.month
spills['Year'] = spills['Date of Discovery'].dt.year

#Clean up county names
spills['county'] = spills['county'].str.strip()

# Fill NaNs with zeros across the volume columns that we care about
nafills = ["Oil BBLs Spilled", "Oil BBLs Recovered", "Produced Water BBLs Spilled",
           "Produced Water BBLs Recovered", "Drilling Fluid BBLs Spilled",
           "Drilling Fluid BBLs Recovered", "Condensate BBLs Spilled",
           "Condensate BBLs Recovered", "Other E&P Waste BBLS Spilled",
           "Other E&P Waste BBLS Recovered", "Flow Back Fluid BBLs Spilled",
           "Flow Back Fluid BBLs Recovered"]
for a in nafills:
  spills[a].fillna(0, inplace=True)

Check the next cell in future years!

- You only need to update it if there are other operators with similar names that need to get consolidated.
- Right now it consolidates multiple names for Anadarko, Kerr McGee, Morningstar, and Noble.

In [10]:
#@title Clean up operators using similar names — INSTRUCTIONS IN THIS CELL -- come back to this later (after dataset is generated, sort by operator name and come back and fix duplicates)

# If you need to add a name to clean up, copy and paste the two lines of code
# that begin with 'mask =' and 'spills.loc' and change the operator name
# to the name of the company you need to consolidate.

##### CHECK THIS THIS IN FUTURE YEARS ######

# If Operator Name starts with "ANADARKO", change it to "ANADARKO"
mask = spills['Operator'].str.startswith('ANADARKO')
spills.loc[mask, 'Operator'] = 'ANADARKO'

# If operator name starts with "KERR MCGEE" change it to "KERR MCGEE"
mask = spills['Operator'].str.startswith('KERR MCGEE')
spills.loc[mask, 'Operator'] = 'KERR MCGEE'

# If operator name starts with "MORNINGSTAR" change it to "MORNINGSTAR"
mask = spills['Operator'].str.startswith('MORNINGSTAR')
spills.loc[mask, 'Operator'] = 'MORNINGSTAR'

# If operator name starts with "NOBLE" change it to "NOBLE"
mask = spills['Operator'].str.startswith('NOBLE')
spills.loc[mask, 'Operator'] = 'NOBLE'

##### CHECK THIS THIS IN FUTURE YEARS ######

In [11]:
#@title Select only discovery dates in this year, recent spills only

mask = (spills['Date of Discovery'] >= f'{year}-01-01') & \
        (spills['Date of Discovery'] <= f'{year}-12-31') & \
        (spills['Spill Type'] == 'Recent')

spills_current = spills.loc[mask]
#how many records do we have now?
print(spills_current.shape)

## The spills_current dataframe is the one we'll use from here on out. (this filters out HISTORIC spills)

(980, 108)


Now you can run the rest of this section!

### Helper functions

#### Dedupe by category

In [12]:
# Dedupe a dataframe based on a unique ID and field. When there's a duplicate,
# pick the most recent entry that doesn't say "Unknown"

def dedupecategory(frame, id, field):

  # Other columns that we care about
  report = "Report"
  operator = "Operator"
  doc = "Document #"

  # only work with rows where the field has something in it
  df = frame[frame[field] != '0']

  # Create a working dataframe with just these columns
  df = df[[id, doc, report, operator, field]]

  # Group them by Tracking number
  grouped = df.groupby(id)

  # Initialize an empty list to store the deduplicated rows
  deduplicated_rows = []

  # Iterate over the grouped dataframe
  for name, group_df in grouped:
      # Initialize a flag to track if the row with the highest document has
      #'field' = 'Unknown'
      unknown_field = False

      # Sort the group dataframe by document in descending order
      group_df = group_df.sort_values(doc, ascending=False)

      # Iterate over the sorted group dataframe
      for index, row in group_df.iterrows():
          # If the field is not 'Unknown'
          if row[field] != 'Unknown':
              # Store the row in the deduplicated_rows list
              deduplicated_rows.append(row)
              #deduplicated_rows = pd.concat([deduplicated_rows, row], ignore_index=True)
              break
          # If the field is 'Unknown', set the flag to True
          else:
              unknown_field = True

      # If the flag is True, meaning all rows have 'field' = 'Unknown'
      if unknown_field:
          # Store the first row in the deduplicated_rows list
          deduplicated_rows.append(group_df.iloc[0])

  # Convert the deduplicated_rows list to a new dataframe and return that
  deduplicated_df = pd.DataFrame(deduplicated_rows)

  return deduplicated_df

#### Dedupe by largest

In [13]:
# Dedupe a dataframe based on a unique ID and field. When there's a duplicate,
# pick the row with the largest entry in that field.

def dedupe_max(frame, id, field):

  # Other columns that we care about
  report = "Report"
  operator = "Operator"

  # create a working dataframe with just these columns
  df = frame[[id, report, operator, field]]

  # Group them by Tracking number
  #print('Grouping by tracking number...')
  grouped = df.groupby(id)

  # Create a new data frame to store the results
  result = pd.DataFrame(columns=df.columns)

  # For each group of duplicate IDs
  for name, group in grouped:
      #print(f'Deduping {name}...')
      # Initialize a new row with the Report, Operator, Date, and Id columns
      # Find the row with the largest value in the column
      row = group.loc[group[field].idxmax()]
      #print(f'Row:\n{row}')

      # Append the row to the result data frame
      #result = result.append(row)

      # Using .concat instead of .append
      #pdb.set_trace()
      #result = pd.concat([result, row])
      result.loc[len(result)] = row

  # Drop rows that are zeroes
  result = result[result[field] > 0]

  # Reset the index of the result data frame
  result.reset_index(drop=True, inplace=True)

  return result

In [14]:
#@title Dedupe, keeping all liquids and adding a sum column

# List of liquids, can be used in any function but especially this one
liquids = ['Oil BBLs Spilled', 'Condensate BBLs Spilled',
           'Produced Water BBLs Spilled',
           'Other E&P Waste BBLS Spilled', 'Flow Back Fluid BBLs Spilled',
           'Drilling Fluid BBLs Spilled']

# Takes a frame and an unique ID from that frame, along with a list of fields.
# Returns a new frame with the largest number from each of
# the fields it looks at.

def dedupe_liquids(df, id, liquids):
  # fields is a list of columns, probably Oil BBLs Spilled, Condensate BBLs Spilled,
  # Produced Water BBLs Spilled, Drilling Fluid BBLs Spilled,
  # Flow Back Fluid BBLs Spilled, Other E&P Waste BBLS Spilled

  # Reset the index of the incoming dataframe
  df = df.reset_index(drop=True)

  # Other columns that we care about
  report = "Report"
  operator = "Operator"
  date = "Date of Discovery"
  date_year = "Year"

  allfields = [id, report, operator, date, date_year]
  allfields.extend(liquids)

  # create a working dataframe with just these columns
  newframe = df[allfields]

  # Group them by Tracking number
  grouped = newframe.groupby(id)

  # Create a new data frame to store the results
  #result = pd.DataFrame(columns=newframe.columns)
  # --- The fix: Initialize result with the 'Total Gallons Spilled' column ---
  result = pd.DataFrame(columns=newframe.columns.tolist() + ['Total Gallons Spilled'])


  # For each group of duplicate IDs
  for name, group in grouped:
    # Initialize a new row with the Report, Operator, Date, and Id columns
    new_row = pd.Series(dtype="object")
    new_row[id] = name
    new_row[report] = group[report].iloc[0]
    new_row[operator] = group[operator].iloc[0]
    new_row[date] = group[date].iloc[0]
    new_row[date_year] = group[date_year].iloc[0]
    total = 0  #for the total column
    # For each column we're going to compare
    for field in liquids:

        # Find the row with the largest value in the column
        #value = group.loc[group[field].idxmax()]
        value = group[field].max()
        # Set the value of that field in our temporary row
        new_row[field] = value
        # And add it to the total column
        total += value

    # Create the total field
    new_row['Total Gallons Spilled'] = gallons(total)
    # Append the row to the result data frame
    # .append is deprecated
    #result = result.append(new_row, ignore_index=True)
    #result = pd.concat([result, pd.DataFrame(new_row)], ignore_index=True)

    # --- Added checks ---
    #print(f"New row: {new_row}")
    #print(f"New row columns: {new_row.index.tolist()}")
    #print(f"Result DataFrame columns: {result.columns.tolist()}")
    #if 'Total Gallons Spilled' not in new_row.index:
    #    raise ValueError("'Total Gallons Spilled' column not found in new_row")
    #if 'Total Gallons Spilled' not in result.columns:
    #    raise ValueError("'Total Gallons Spilled' column not found in result DataFrame")
    # --- End of added checks ---

    # Fixing broken pd.concat
    result = pd.concat([result,
                        pd.DataFrame([new_row],
                        columns=result.columns)],
                        ignore_index=True) # Changed to use concat with explicit columns

    # Reset the index of the result data frame
    result.reset_index(drop=True, inplace=True)

  return result


In [15]:
# # Moar testing!

# liquids = ['Oil BBLs Spilled', 'Condensate BBLs Spilled',
#            'Produced Water BBLs Spilled',
#            'Other E&P Waste BBLS Spilled', 'Flow Back Fluid BBLs Spilled',
#            'Drilling Fluid BBLs Spilled']

# # Deduplicate
# deduped = dedupe_liquids(df, id, liquids)


#### Dedupe by smallest

In [16]:
# Dedupe a dataframe based on a unique ID and field. When there's a duplicate,
# pick the row with the smallest entry in that field.

# Dedupe a dataframe based on a unique ID and field. When there's a duplicate,
# pick the row with the largest entry in that field.

def dedupe_min(frame, id, field):

  # Other columns that we care about
  report = "Report"
  operator = "Operator"

  # create a working dataframe with just these columns
  df = frame[[id, report, operator, field]]

  # Group them by Tracking number
  grouped = df.groupby(id)

  # Create a new data frame to store the results
  result = pd.DataFrame(columns=df.columns)

  # For each group of duplicate IDs
  for name, group in grouped:
      # Find the row with the largest value in the column
      row = group.loc[group[field].idxmin()]

      # Append the row to the result data frame
      # result = result.append(row) #.append is deprecated
      #result = pd.concat([result, row])
      result.loc[len(result)] = row

  # Drop rows that are zeroes
  result = result[result[field] > 0]

  # Reset the index of the result data frame
  result.reset_index(drop=True, inplace=True)

  return result

#### Summarize by volume, write to file

In [17]:
# For making a summary table that dedupes then adds a field
# And appends the output to a specific file.
# Also returns a total volume spilled in gallons.

# desc is a string describing the field, eg "oil" or "produced water"
# filename has *no extension* on it
def summarize(df, id, field, filename, desc):

  # Run the de-dupe
  deduped = dedupe_max(df, id, field)
  #print('Deduped table:')
  #print(deduped)

  # add it up
  sum = deduped[field].sum()
  string = ("Total " + str(desc) + " spilled:\n" + str(sum) + " barrels\n" + str(gallons(sum)) + " gallons\n\n")
  print(string)

  # add it to our text file
  textfile = (f'{workingdir}/colorado/{filename}.txt')
  with open(textfile, "a") as file:
    file.write(string)

  return gallons(sum)

#### Summarize by group, write to file


In [18]:
# For making a summary table that dedupes then groups a field
# And appends the output to a sepcific file
# Returns a total number of spills as well

# desc is a string describing the field in past tense, eg "oil"
# filename has *no extension* on it
def groupcount(df, id, field, filename, desc):

  # Run the dedupe
  categories = dedupecategory(spills_current, id, field)
  # Make a pivot table
  pivot = pd.pivot_table(categories, index = field, \
                         values = id, aggfunc="nunique")
  sum = pivot[id].sum()

  # Cut off the header rows from the pivot table
  split = str(pivot).split("\n")
  split = split[2:]
  joined = "\n".join(split)

  # Make a nice long string for output

  out = ("Number of " + str(desc) + " spills: " + str(sum))
  out += ("\n\n")
  out += (joined + "\n\n")

  # Add it to our text file, print it
  textfile = (f'{workingdir}/colorado/{filename}.txt')
  with open(textfile, "a") as file:
    file.write(out)

  print(out)

  # Also make a unique csv file and put just the pivot table there
  csvfile = (f'{workingdir}/colorado/{filename}.csv')
  pivot.to_csv(csvfile)

  return sum

#### Summarize by distance

In [19]:
# Summarize a column by distance, putting in buckets of 500.
# Returns a pivot table with the buckets.

def distancegroup(frame, id, field):
  # drop rows with no number in our field
  df = spills_current.dropna(subset=[field])

  # Now dedupe, taking the smallest distance for each duplicated tracking number
  df = dedupe_min(df, id, field)

  # Group them by distance, in buckets.
  bins = [0, 501, 1001, 5281]
  labels = ['0-500', '501-1000', '1001-5280']

  # categorize the column using the `cut` function
  df['distance'] = pd.cut(df[field], bins=bins, labels=labels, include_lowest=True)

  # Make that pivot table
  pivot = pd.pivot_table(df, index = ["distance"], \
                                values = "Tracking #", aggfunc="nunique")

  return pivot


#### Create the master report file

In [20]:
reportfile = (f'{workingdir}/colorado/colorado_report.txt')
with open(reportfile, "w") as file:
  string = (hr +
            "COLORADO " + str(year) + " SPILLS REPORT\n" +
            hr + "\n")
  file.write(string)

print("File " + reportfile + " created.")

File /content/data/colorado/colorado_report.txt created.


In [21]:
#@title Create a dataframe for counting all substances

allcounts = pd.DataFrame(columns=['Number of Spills', 'Total Spill Volume'])

## Pivot tables and summaries


In [22]:
#@title Make pivot tables by volume spilled, using range buckets for oil and produced water

# Make a dataframe df based on spills_current where "Oil Spill Volume" is not 0
#df = spills_current[spills_current["Oil Spill Volume"] != 0]

# Make a pivot table called oil_buckets_pivot. Use spills_current, excluding rows where "Oil Spill Volume" is 0. Index is "Oil Spill Volume", Values are "Tracking #", aggfunc "nunique"
oil_buckets_pivot = pd.pivot_table(spills_current[spills_current["Oil Spill Volume"] != "0"],
                                   index = ["Oil Spill Volume"],
                                   values = "Tracking #", aggfunc="nunique")

# write the table to a csv file, oil_spilled_groups.csv
oil_buckets_pivot.to_csv(f'{workingdir}/colorado/oil_spilled_groups.csv')

# Add a total row to oil_buckets_pivot
sum = oil_buckets_pivot.sum()

# make a pretty multi-line text string
out = (hr + "BY SIZE OF OIL SPILL\n" + hr)
out += ("Total oil spills: " + str(sum) + "\n")
out += (oil_buckets_pivot.to_string() + "\n\n")

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)

# Now do the same thing but make produced_buckets_pivot, on "Produced Water Spill Volume"
produced_buckets_pivot = pd.pivot_table(spills_current[spills_current["Produced Water Spill Volume"] != "0"],
                                   index = ["Produced Water Spill Volume"],
                                   values = "Tracking #", aggfunc="nunique")

# write the table to a csv file, produced_spilled_groups.csv
produced_buckets_pivot.to_csv(f'{workingdir}/colorado/produced_spilled_groups.csv')

# Add a total row to produced_buckets_pivot
sum = produced_buckets_pivot.sum()

# make a pretty multi-line text string
out = (hr + "BY SIZE OF PRODUCED WATER SPILL\n" + hr)
out += ("Total produced water spills: " + str(sum) + "\n")
out += (produced_buckets_pivot.to_string() + "\n\n")

# Append all of it to our summary report and print
with open(reportfile, "a") as file:
  file.write(out)

print(out)


---------------
BY SIZE OF OIL SPILL
---------------
Total oil spills: Tracking #    155
dtype: int64
                  Tracking #
Oil Spill Volume            
>0 and <1                 30
>=1 and <5                32
>=100                      1
>=5 and <100              35
Unknown                   57


---------------
BY SIZE OF PRODUCED WATER SPILL
---------------
Total produced water spills: Tracking #    256
dtype: int64
                             Tracking #
Produced Water Spill Volume            
>0 and <1                            16
>=1 and <5                           47
>=100                                13
>=5 and <100                         74
Unknown                             106




In [23]:
#@title Make a Pivot Table by operator, counting unique tracking numbers
operators_pivot = pd.pivot_table(spills_current, index = ["Operator"], \
                                 values = ["Tracking #"], aggfunc = "nunique") \
                                 .sort_values(by="Tracking #", ascending=False)

# write the table to a csv file
operators_pivot.to_csv(f'{workingdir}/colorado/operators.csv')

# add them all up
sum = operators_pivot["Tracking #"].sum()
total_spill_count = sum

# Take the top 5, add up the rest into an "other" row
top_5 = operators_pivot.nlargest(5, id)
# Calculate the total of top 5 as percentage of all spills
top_5_sum = top_5["Tracking #"].sum()
top_5_pct =  "{:.0%}".format(top_5_sum / total_spill_count)

# make an "others" pivot table, add it up and rename it
others = operators_pivot[~operators_pivot.index.isin(top_5.index)].sum().rename('OTHER')
result = pd.concat([top_5, others.to_frame().T])
result.to_csv(f'{workingdir}/colorado/co operators_top5 {year}.csv')

topten = str(result).split("\n")
topten = topten[1:]
joined = "\n".join(topten)

# make a pretty multi-line text string
out = (hr + "BY OPERATOR\n" + hr)
out += ("Total spills: " + str(sum) + "\n")
out += (f'The top 5 operators were responsible for {top_5_pct} percent of all spills.\n\n')
out += ("Top 5 operators:\n")
out += (joined + "\n\n")


# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)


---------------
BY OPERATOR
---------------
Total spills: 342
The top 5 operators were responsible for 45% percent of all spills.

Top 5 operators:
QB ENERGY OPERATING LLC          46
KERR MCGEE                       35
PDC ENERGY INC                   30
NOBLE                            29
TEP ROCKY MOUNTAIN LLC           15
OTHER                           187




In [24]:
#@title Make a Pivot Table by surface owner, counting unique tracking numbers
df = spills_current[spills_current['Surface Owner'] != '0']
surface_pivot = pd.pivot_table(df, index = ["Surface Owner"], \
                              values = "Tracking #", aggfunc="nunique")
surface_pivot = surface_pivot.sort_values(by=['Tracking #'], ascending=False)

# Rename our rows
surface_pivot = surface_pivot.rename(index={'FEE': 'Private',
                                            'OTHER (SPECIFY)': 'Other',
                                            'FEDERAL': 'Federal',
                                            'STATE': 'State'})

#write the table to a text file
surface_pivot.to_csv(f'{workingdir}/colorado/co where {year}.csv')

# make a pretty output string, save it and print it
sum = surface_pivot["Tracking #"].sum()
split = str(surface_pivot).split("\n")
split = split[2:]
joined = "\n".join(split)

out = (hr + "BY SURFACE OWNER\n" + hr)
out += ("Total spills: " + str(sum) + "\n\n")
out += (joined + "\n\n")

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
BY SURFACE OWNER
---------------
Total spills: 340

Private               276
Federal                33
Other                  23
State                   7
TRIBAL                  1




In [25]:
#@title By county

# List with all counties, 0 for all of them
county_names_mixedcase = ["Adams", "Alamosa", "Arapahoe", "Archuleta", "Baca", "Bent", "Boulder", "Broomfield", "Chaffee", "Cheyenne", "Clear Creek", "Conejos", "Costilla", "Crowley", "Custer", "Delta", "Denver", "Dolores", "Douglas", "Eagle", "El Paso", "Elbert", "Fremont", "Garfield", "Gilpin", "Grand", "Gunnison", "Hinsdale", "Huerfano", "Jackson", "Jefferson", "Kiowa", "Kit Carson", "La Plata", "Lake", "Larimer", "Las Animas", "Lincoln", "Logan", "Mesa", "Mineral", "Moffat", "Montezuma", "Montrose", "Morgan", "Otero", "Ouray", "Park", "Phillips", "Pitkin", "Prowers", "Pueblo", "Rio Blanco", "Rio Grande", "Routt", "Saguache", "San Juan", "San Miguel", "Sedgwick", "Summit", "Teller", "Washington", "Weld", "Yuma"]
# County coordinates from Infogram
county_coordinates = ["39.8706 -103.8766", "37.5523 -105.8429", "39.652 -104.4353", "37.2084 -107.2676", "37.3198 -102.5601", "37.9549 -103.0734", "40.0925 -105.3587", "39.9547 -105.0534", "38.7468 -106.1936", "38.8297 -102.5211", "39.7084 -105.6625", "37.1987 -106.1984", "37.2786 -105.4271", "38.3182 -103.7798", "38.0778 -105.4231", "38.8594 -107.8659", "39.7643 -104.96", "37.7523 -108.5192", "39.3294 -104.9279", "39.6374 -106.6449", "38.8247 -104.5621", "39.2866 -104.1361", "38.4776 -105.4766", "39.7288 -107.2377", "39.8585 -105.5229", "40.1025 -106.1183", "38.7022 -107.0797", "37.8212 -107.301", "37.654 -104.9264", "40.6658 -106.3416", "39.586 -105.2505", "38.4401 -102.7758", "39.3058 -102.6049", "37.2866 -107.8427", "39.2015 -106.345", "40.6667 -105.4609", "37.3158 -104.0371", "38.9886 -103.5155", "40.7193 -103.1163", "38.9333 -108.6266", "37.6163 -106.9211", "40.6118 -107.877", "37.3192 -108.7255", "38.4023 -108.2703", "40.2626 -103.8074", "37.9028 -103.7162", "38.1549 -107.769", "39.1189 -105.7177", "40.594 -102.51", "39.1719 -106.8807", "37.9562 -102.3948", "38.1736 -104.5126", "39.9378 -108.7634", "37.583 -106.383", "40.4854 -106.991", "38.0808 -106.282", "37.7645 -107.6755", "37.9633 -108.0935", "40.876 -102.3518", "39.6406 -106.108", "38.8816 -105.1624", "39.9708 -103.2009", "40.5011 -104.5558", "39.9447 -102.4267"]
county_names = [s.upper() for s in county_names_mixedcase] # Need upper case for matching to dataset
counties = pd.DataFrame({
    'county': county_names,
    'coordinates': county_coordinates,
    'group': '',
    'label': county_names
})
#counties = pd.DataFrame(data=county_names, index=range(0,64), columns=['county'])
counties['Tracking #'] = 0
counties['county'] = counties['county'].str.strip()
counties['label'] = counties['county'].str.strip()

# Make a Pivot Table by county, counting unique tracking numbers
df = spills_current[spills_current['county'] != '0']
county_pivot = pd.pivot_table(df, index = ["county"], \
                              values = "Tracking #", aggfunc="nunique")

# write the table to a text file
county_pivot.to_csv(f'{workingdir}/colorado/county.csv')

# merge it with our county list
merged = counties.merge(county_pivot, right_on='county', left_on='county', how='outer')
merged = merged.drop('Tracking #_x', axis=1)
merged = merged.fillna(0).rename(columns={'Tracking #_y': 'Spill Count'})

# Save that to a CSV, in the column order for Infogram
merged = merged[['county', 'Spill Count', 'group', 'coordinates', 'label']]
merged['county'] = county_names_mixedcase # Back to mixed case for Infogram
merged['label'] = county_names_mixedcase
merged['Spill Count'] = merged['Spill Count'].astype(int) # Converting to integer for output
merged.to_csv(f'{workingdir}/colorado/co allcounties {year}.csv', index=False, header=False)

# make a pretty output string, save it and print it
sum = county_pivot["Tracking #"].sum()
split = str(county_pivot).split("\n")
split = split[2:]
joined = "\n".join(split)

out = (hr + "BY COUNTY\n" + hr)
out += ("Total spills: " + str(sum) + "\n\n")
out += (joined + "\n\n")

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)


---------------
BY COUNTY
---------------
Total spills: 338

ADAMS               20
ARAPAHOE             3
CHEYENNE             6
ELBERT               1
GARFIELD            61
GUNNISON             1
JACKSON              9
KIOWA                2
LA PLATA            11
LARIMER              1
LAS ANIMAS           8
LOGAN                2
MESA                 7
MOFFAT               4
MONTEZUMA            2
PHILLIPS             1
RIO BLANCO          19
SAN MIGUEL           1
WASHINGTON           2
WELD               170
YUMA                 7




In [26]:
#@title Make a Pivot Table by month, counting unique tracking numbers
df = spills_current
month_pivot = pd.pivot_table(df, index = ["Month"], \
                                values = "Tracking #", aggfunc="nunique")
sum = month_pivot["Tracking #"].sum()
# Map month names to month numbers
month_map = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
             7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
month_pivot.index = month_pivot.index.map(month_map)

#write the table to a text file
month_pivot.to_csv(f'{workingdir}/colorado/month.csv')

# make a pretty output string, save it and print it
sum = month_pivot["Tracking #"].sum()
split = str(month_pivot).split("\n")
split = split[2:]
joined = "\n".join(split)

out = (hr + "BY MONTH\n" + hr)
out += ("Total spills: " + str(sum) + "\n\n")
out += (joined + "\n\n")

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
BY MONTH
---------------
Total spills: 338

January            48
February           35
March              28
April              31
May                20
June               27
July               27
August             28
September          22
October            22
November           29
December           21




### TODO NEXT YEAR
- Combine the next several cells into one cell, running over a list of `filename` and `substance` tuples

In [ ]:
#@title 2025 UPDATE NEEDED: Summarize and calculate oil spill volume
## Need to calculate number of spills with unknown spill volumes

filename = 'oilspills'
substance = "oil"

# Initialize the text file
textfile = f'{workingdir}/colorado/{filename}.txt'
with open(textfile, "w") as file:
  file.write(hr + "Oil Spills\n" + hr)

# Do the summaries, which write to the file
try:
  spillcount = groupcount(spills_current, id, 'Oil Spill Volume', filename, substance)
except KeyError:
  spillcount = 0
  with open(filename, "w") as file:
    print("None reported")
try:
  spilltotal = summarize(spills_current, id, 'Oil BBLs Spilled', filename, substance)
except KeyError:
  spilltotal = 0
  with open(filename, "w") as file:
    print("None reported")

allcounts.loc[substance] = [spillcount, spilltotal]

# append the text file we just made to the summary report
with open(textfile, "r") as f1:
  text1 = f1.read()

with open(reportfile, "a") as f2:
  f2.write(text1)

Number of oil spills: 167

>0 and <1                 36
>=1 and <5                44
>=100                      6
>=5 and <100              23
Unknown                   58


Total oil spilled:
1532.0 barrels
64344.0 gallons




In [ ]:
#@title Summarize and calculate condensate spill volume
filename = 'condensatespills'
substance = "condensate"

# Initialize the text file
textfile = f'{workingdir}/colorado/{filename}.txt'
with open(textfile, "w") as file:
  file.write(hr + "Condensate Spills\n" + hr)

# Do the summaries, which write to the file
try:
  spillcount = groupcount(spills_current, id, 'Condensate Spill Volume', filename, substance)
except KeyError:
  spillcount = 0
  with open(filename, "w") as file:
    print("None reported")

try:
  spilltotal = summarize(spills_current, id, 'Condensate BBLs Spilled', filename, substance)
except KeyError:
  spilltotal = 0
  with open(filename, "w") as file:
    print("None reported")

allcounts.loc[substance] = [spillcount, spilltotal]


# append the text file we just made to the summary report
with open(textfile, "r") as f1:
  text1 = f1.read()

with open(reportfile, "a") as f2:
  f2.write(text1)

Number of condensate spills: 72

>0 and <1                        12
>=1 and <5                        4
>=5 and <100                      3
Unknown                          53


Total condensate spilled:
271.0 barrels
11382.0 gallons




In [ ]:
#@title Summarize and calculate produced water spill volume
filename = 'producedwaterspills'
substance = "produced water"

# Initialize the text file
textfile = f'{workingdir}/colorado/{filename}.txt'
with open(textfile, "w") as file:
  file.write(hr + "Produced Water Spills\n" + hr)

# Do the summaries, which write to the file
try:
  spillcount = groupcount(spills_current, id, 'Produced Water Spill Volume', filename, substance)
except KeyError:
  spillcount = 0
  with open(filename, "w") as file:
    print("None reported")

try:
  spilltotal = summarize(spills_current, id, 'Produced Water BBLs Spilled', filename, substance)
except KeyError:
  spilltotal = 0
  with open(filename, "w") as file:
    print("None reported")

allcounts.loc[substance] = [spillcount, spilltotal]

# append the text file we just made to the summary report
with open(textfile, "r") as f1:
  text1 = f1.read()

with open(reportfile, "a") as f2:
  f2.write(text1)

Number of produced water spills: 260

>0 and <1                            22
>=1 and <5                           46
>=100                                14
>=5 and <100                         80
Unknown                              98


Total produced water spilled:
5856.0 barrels
245952.0 gallons




In [ ]:
#@title Summarize and calculate E&P waste spill volume
filename = 'otherepwaste'
substance = "other E&P waste"

# Initialize the text file
textfile = f'{workingdir}/colorado/{filename}.txt'
with open(textfile, "w") as file:
  file.write(hr + "Other E&P Waste Spills\n" + hr)

# Do the summaries, which write to the file
try:
  spillcount = groupcount(spills_current, id, 'E&P Waste Spill Volume', filename, substance)
except KeyError:
  spillcount = 0
  with open(filename, "w") as file:
    print("None reported")

# Do the summaries, which write to the file
try:
  spilltotal = summarize(spills_current, id, 'Other E&P Waste BBLS Spilled', filename, substance)
except KeyError:
  spilltotal = 0
  with open(filename, "w") as file:
    print("None reported")

allcounts.loc[substance] = [spillcount, spilltotal]

# append the text file we just made to the summary report
with open(textfile, "r") as f1:
  text1 = f1.read()

with open(reportfile, "a") as f2:
  f2.write(text1)

Number of other E&P waste spills: 27

>0 and <1                        3
>=1 and <5                       2
>=5 and <100                     4
Unknown                         18


Total other E&P waste spilled:
2641.0 barrels
110922.0 gallons




In [ ]:
#@title Summarize and calculate flow back spill volume
filename = 'flowbackspills'
substance = "flow back fluid"

# Initialize the text file
textfile = f'{workingdir}/colorado/{filename}.txt'
with open(textfile, "w") as file:
  file.write(hr + "Flow Back Fluid Spills\n" + hr)

# Do the summaries, which write to the file
try:
  spillcount = groupcount(spills_current, id, 'Flow Back Spill Volume', filename, substance)
except KeyError:
  spillcount = 0
  with open(filename, "w") as file:
    print("None reported")

try:
  spilltotal = summarize(spills_current, id, 'Flow Back Fluid BBLs Spilled', filename, substance)
except KeyError:
  spilltotal = 0
  with open(filename, "w") as file:
    print("None reported")

allcounts.loc[substance] = [spillcount, spilltotal]

# append the text file we just made to the summary report
with open(textfile, "r") as f1:
  text1 = f1.read()

with open(reportfile, "a") as f2:
  f2.write(text1)

Number of flow back fluid spills: 3

>=5 and <100                     2
Unknown                          1


Total flow back fluid spilled:
9.0 barrels
378.0 gallons




In [ ]:
#@title Summarize and calculate drilling fluid spill volume
filename = 'drillingspills'
substance = "drilling fluid"

# Initialize the text file
textfile = f'{workingdir}/colorado/{filename}.txt'
with open(textfile, "w") as file:
  file.write(hr + "Drilling Fluid Spills\n" + hr)

# Do the summaries, which write to the file
try:
  spillcount = groupcount(spills_current, id, 'Drilling Fluid Spill Volume', filename, substance)
except KeyError:
  spillcount = 0
  with open(filename, "w") as file:
    print("None reported")

try:
  spilltotal = summarize(spills_current, id, 'Drilling Fluid BBLs Spilled', filename, substance)
except KeyError:
  spilltotal = 0
  with open(filename, "w") as file:
    print("None reported")

allcounts.loc[substance] = [spillcount, spilltotal]

# append the text file we just made to the summary report
with open(textfile, "r") as f1:
  text1 = f1.read()

with open(reportfile, "a") as f2:
  f2.write(text1)

Number of drilling fluid spills: 6

>0 and <1                             2
>=1 and <5                            1
>=5 and <100                          1
Unknown                               2


Total drilling fluid spilled:
75.0 barrels
3150.0 gallons




### TODO NEXT YEAR
- Combine the distance calculations into one cell, looping over a list of fields.

In [ ]:
#@title Distance to surface water

# field that we care about
field = "Surface Water Near"

#only look at spills less than a mile away
lessthanamile = spills_current[spills_current[field] <= 5280]

# do the pivot
water_pivot = distancegroup(lessthanamile, id, field)

# write the table to a text file
water_pivot.to_csv(f'{workingdir}/colorado/co surfacewater {year}.csv')

# make a pretty output string, save it and print it
sum = water_pivot["Tracking #"].sum()
split = str(water_pivot).split("\n")
split = split[2:]
joined = "\n".join(split)

out = (hr + "DISTANCE TO SURFACE WATER (feet)\n" + hr)
out += ("Total spills: " + str(sum) + "\n")
pct_reporting = "{:.1%}".format(sum / total_spill_count)
out += (f'Percentage of spills reporting distance: {pct_reporting}\n\n')
out += (joined + "\n\n")

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)


---------------
DISTANCE TO SURFACE WATER (feet)
---------------
Total spills: 299
Percentage of spills reporting distance: 78.1%

0-500              71
501-1000           67
1001-5280         161




In [ ]:
#@title Distance to water wells

# field that we care about
field = "Water Wells"

#only look at spills less than a mile away
lessthanamile = spills_current[spills_current[field] <= 5280]

# do the pivot
well_pivot = distancegroup(lessthanamile, id, field)

# write the table to a text file
well_pivot.to_csv(f'{workingdir}/colorado/co waterwells {year}.csv')

# make a pretty output string, save it and print it
sum = well_pivot["Tracking #"].sum()
split = str(well_pivot).split("\n")
split = split[2:]
joined = "\n".join(split)

out = (hr + "DISTANCE TO WATER WELL (feet)\n" + hr)
out += ("Total spills: " + str(sum) + "\n\n")
out += (joined + "\n\n")
pct_reporting = "{:.0%}".format(sum / total_spill_count)
out += (f'Around {pct_reporting} of spills included data on their proximity to a water well.')


# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)


---------------
DISTANCE TO WATER WELL (feet)
---------------
Total spills: 300

0-500              30
501-1000           79
1001-5280         191

Around 78% of spills included data on their proximity to a water well.


In [ ]:
#@title Distance to occupied buildings

# field that we care about
field = "Occupied Buildings"

#only look at spills less than a mile away
lessthanamile = spills_current[spills_current[field] <= 5280]

# do the pivot
building_pivot = distancegroup(lessthanamile, id, field)

# write the table to a text file
building_pivot.to_csv(f'{workingdir}/colorado/co buildings {year}.csv')

# make a pretty output string, save it and print it
sum = building_pivot["Tracking #"].sum()
split = str(building_pivot).split("\n")
split = split[2:]
joined = "\n".join(split)

out = (hr + "DISTANCE TO OCCUPIED BUILDING (feet)\n" + hr)
out += ("Total spills: " + str(sum) + "\n\n")
pct_reporting = "{:.0%}".format(sum / total_spill_count)
out += (f'Around {pct_reporting} of spills included data on their proximity to an occupied building.\n')
out += (joined + "\n\n")

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)


---------------
DISTANCE TO OCCUPIED BUILDING (feet)
---------------
Total spills: 235

Around 61% of spills included data on their proximity to an occupied building.
0-500              44
501-1000           53
1001-5280         138




## Comparisons to previous years

In [27]:
#@title Number of spills per year
#Get everything since 2014, and through the year we're analyzing (exclude the current partial year)
mask = (spills['Date of Discovery'] >= f'2014-01-01') & \
        (spills['Date of Discovery'] <= f'{year}-12-31') & \
        (spills['Spill Type'] == 'Recent')
spills_recent = spills.loc[mask]

# Get a total number of spills per year
df = spills_recent
year_pivot = pd.pivot_table(df, index = ["Year"], \
                                values = "Tracking #", aggfunc="nunique")
sum = year_pivot["Tracking #"].sum()

#save it to a csv
year_pivot.to_csv(f'{workingdir}/colorado/spills_years.csv')

out = (f'{hr}Spills per year\n{hr}')
split = str(year_pivot).split("\n")
split = split[2:]
joined = "\n".join(split)
out += (joined)
out += (f'\n\nTotal spills since 2014: {comma.format(sum)}')

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print (out)

---------------
Spills per year
---------------
2014           2
2016         248
2017         437
2018         400
2019         455
2020         349
2021         409
2022         457
2023         419
2024         378
2025         338

Total spills since 2014: 3,892


In [28]:
#@title Amount spilled

# add up these liquid columns
liquids = ['Oil BBLs Spilled',
           'Condensate BBLs Spilled',
           'Produced Water BBLs Spilled',
           'Other E&P Waste BBLS Spilled',
           'Flow Back Fluid BBLs Spilled',
           'Drilling Fluid BBLs Spilled']

# Empty series that will hold our results
totals = pd.Series(dtype='float')

# add up the total volume spilled in each of the last 5 years
for thisyear in range(year-4, year+1, 1):
  #just this year
  print(f'Analyzing {thisyear}...')
  mask = (spills['Date of Discovery'] >= f'{thisyear}-01-01') & \
        (spills['Date of Discovery'] <= f'{thisyear}-12-31') & \
        (spills['Spill Type'] == 'Recent')
  df = spills.loc[mask]
  #initialize our sum
  thisvolume = 0
  # Do a dedupe and add the total volume together for every type of liquid
  for liquid in liquids:
    deduped = dedupe_max(df, id, liquid)
    thisvolume += deduped[liquid].sum()
  # add the results to the series, in gallons
  totals[f'{thisyear}'] = thisvolume * 42

#  add them all up
total_years = totals.sum()

# Save it and print it, add it to the report
totals.to_csv(f'{workingdir}/colorado/volume_years.csv')

out = (f'{hr}Volume spilled per year (gallons)\n{hr}')
split = str(totals).split("\n")
split = split[:-1]
joined = "\n".join(split)
out += (joined)
out += (f'\n\nTotal gallons spilled, {year-4}-{year}: {comma.format(total_years)}')

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)

print(out)




Analyzing 2021...
Analyzing 2022...
Analyzing 2023...
Analyzing 2024...
Analyzing 2025...
---------------
Volume spilled per year (gallons)
---------------
2021     723,744
2022   1,460,340
2023   1,050,714
2024     436,128
2025     794,472

Total gallons spilled, 2021-2025: 4,465,398


In [29]:
#@title Operators with biggest increases/decreases, year to year

# List of liquids is defined in the helper function cell up at the top

# get two years of results
print(f'Analyzing {lastyear}-{year}...')
mask = (spills['Date of Discovery'] >= f'{lastyear}-01-01') & \
      (spills['Date of Discovery'] <= f'{year}-12-31') & \
      (spills['Spill Type'] == 'Recent')
df = spills.loc[mask].copy()

# Deduplicate
deduped = dedupe_liquids(df, id, liquids)

#temporary check
#deduped.to_csv('./spills2years.csv')

#Add that to the columns in the pivot table
#liquids.append('Total gallons spilled')
# pivot by operator and year
pivot = pd.pivot_table(deduped, index = ["Operator"],
                       columns = ["Year"],
                       values = 'Total Gallons Spilled',
                       aggfunc = "sum")

# get the difference between years
pivot['Difference'] = pivot[year] - pivot[lastyear]

# TODO get the percentage difference between years
pivot['Pct Change'] = pivot['Difference'] / pivot[lastyear]

# Save it to a csv
pivot.to_csv(f'{workingdir}/colorado/operator_volume_years.csv')

# Get the top 5 in the current year, just export that
top5 = pivot.sort_values(by=year, ascending=False).head(5).reindex(columns=[year, lastyear, 'Difference', 'Pct Change'])
top5.to_csv(f'{workingdir}/colorado/operator_volume_years_top.csv')

out = ""

# the biggest spillers of the current year
top = pivot[year].nlargest(5)
out = (f'{hr}5 biggest spillers of {year}\n(Gallons)\n{hr}')
split = str(top).split("\n")
split = split[1:-1]
joined = "\n".join(split)
out += joined + '\n\n'

# get the top 10
top = pivot['Difference'].nlargest(5)
out += (f'{hr}5 biggest increases in spills, {lastyear}-{year}\n(Gallons)\n{hr}')
split = str(top).split("\n")
split = split[1:-1]
joined = "\n".join(split)
out += joined + '\n\n'

# get the bottom 10
bottom = pivot['Difference'].nsmallest(5)
out += (f'{hr}5 biggest decreases in spills, {lastyear}-{year}\n(Gallons)\n{hr}')
split = str(bottom).split("\n")
split = split[1:-1]
joined = "\n".join(split)
out += joined + '\n\n'
print(out)

# Append all of it to our summary report
with open(reportfile, "a") as file:
  file.write(out)



Analyzing 2024-2025...
---------------
5 biggest spillers of 2025
(Gallons)
---------------
QB ENERGY OPERATING LLC       445,284
TEP ROCKY MOUNTAIN LLC         67,284
SCOUT ENERGY MANAGEMENT LLC    57,078
LARAMIE ENERGY LLC             42,210
NGL WATER SOLUTIONS DJ LLC     31,458

---------------
5 biggest increases in spills, 2024-2025
(Gallons)
---------------
TEP ROCKY MOUNTAIN LLC       63,084
LARAMIE ENERGY LLC           33,726
NGL WATER SOLUTIONS DJ LLC   16,758
NOBLE                        12,054
DCP OPERATING COMPANY LP      3,864

---------------
5 biggest decreases in spills, 2024-2025
(Gallons)
---------------
SCOUT ENERGY MANAGEMENT LLC               -98,448
CAERUS PICEANCE LLC                       -48,804
FUNDARE RESOURCES OPERATING COMPANY LLC   -16,506
CITATION OIL & GAS CORP                   -11,718
FULCRUM ENERGY OPERATING LLC               -7,056




In [30]:
#@title Operators by number of spills year to year
# List of liquids is defined in the helper function cell up at the top

# get two years of results
print(f'Analyzing {lastyear}-{year}...')
mask = (spills['Date of Discovery'] >= f'{lastyear}-01-01') & \
      (spills['Date of Discovery'] <= f'{year}-12-31') & \
      (spills['Spill Type'] == 'Recent')
df = spills.loc[mask].copy()

# Deduplicate
#deduped = dedupe_liquids(df, id, liquids)

#temporary check
#deduped.to_csv('./spills2years.csv')

#Add that to the columns in the pivot table
#liquids.append('Total gallons spilled')
# pivot by operator and year
pivot = pd.pivot_table(df, index = ["Operator"],
                       columns = ["Year"],
                       values = 'Tracking #',
                       aggfunc = "nunique")

# get the difference between years
pivot['Difference'] = pivot[year] - pivot[lastyear]

# get the percentage difference between years
pivot['Pct Change'] = (pivot['Difference'] / pivot[lastyear])

# Save it to a csv
pivot.to_csv(f'{workingdir}/colorado/co operator_spills_years {year}.csv')

# Get the top 5 in the current year, just export that
# Pretty format the percent column
pivot['Pct Change'] = pivot['Pct Change'].map(lambda x: "{:.0%}".format(x))
top5 = pivot.sort_values(by=year, ascending=False).head(5).reindex(columns=[year, lastyear, 'Pct Change'])
top5.to_csv(f'{workingdir}/colorado/co operator_spills_years_top {year}.csv')

out = (f'{hr}Number of spills\n(year-to-year comparison)\nBy operator\n{hr}')
out += (f'{top5}\n\n')

print(out)
with open(reportfile, "a") as file:
  file.write(out)

Analyzing 2024-2025...
---------------
Number of spills
(year-to-year comparison)
By operator
---------------
Year                     2025  2024 Pct Change
Operator                                      
QB ENERGY OPERATING LLC    46     2      2200%
KERR MCGEE                 35    50       -30%
PDC ENERGY INC             30    32        -6%
NOBLE                      29    27         7%
TEP ROCKY MOUNTAIN LLC     15     1      1400%




In [31]:
#@title Export for the map

# start with the deduped table from the previous cell, add lat/long, county
# do a lookup on spills_current
#spills_current.columns

# Create a series that maps id to latitude from spills_current
nodupes = spills_current.drop_duplicates(subset=id, keep='first')

# Create a list of fields to look up
fields = ['county', 'Facility Type', 'Surface Owner',
          'Spill Description', 'Water Wells',
          'Surface Water','Occupied Buildings',
          'Latitude', 'Longitude']

# Only get spills for the current year
current = deduped[deduped['Year'] == year].copy()

# Use a loop to create a series for each field and map it to deduped
for field in fields:
  id_to_field = nodupes.set_index(id)[field]
  current[field] = current[id].map(id_to_field)

# Drop the Report column
current = current.drop('Report', axis=1)

# Save it
current.to_csv(f'{workingdir}/colorado/mapdata.csv', index=False)

## Output and download

In [34]:
allcounts = allcounts.rename(index={'produced water': 'Produced Water',
                                    'oil': 'Oil',
                                    'condensate': 'Condensate',
                                    'other E&P waste': 'Other E&P Waste',
                                    'drilling fluid': 'Drilling Fluid',
                                    'flow back fluid': 'Flowback Fluid'
                                    })

# save the allcounts dataframe
allcounts = allcounts.sort_values(by='Number of Spills', ascending=False)
allcounts.loc['Total'] = [allcounts['Number of Spills'].sum(), allcounts['Total Spill Volume'].sum()]
allcounts.to_csv(f'{workingdir}/colorado/co allcounts {year}.csv')
allcounts_total = allcounts.loc['Total']
print(f'Total spills:\n{allcounts_total}')

# Save the 'How Many' spills CSV, dropping the Total line
howmany = allcounts['Number of Spills']
howmany = howmany.drop('Total', axis=0)
# Convert to integers
howmany_int = howmany.astype(int)
howmany_int.to_csv(f'{workingdir}/colorado/co howmany {year}.csv')

# Re-sort by volume, save 'How Much' CSV, dropping the Total line
howmuch = allcounts.sort_values(by='Total Spill Volume', ascending=False)
howmuch = howmuch['Total Spill Volume']
howmuch_total = howmuch.loc['Total']
#print(f'Total gallons spilled: {howmuch_total}')
howmuch = howmuch.reindex(['Produced Water', 'Other E&P Waste', 'Oil', 'Condensate', 'Drilling Fluid', 'Flowback Fluid'])
# Fill NaN values with 0 before converting to integers
howmuch = howmuch.fillna(0)
# Convert to integers
howmuch_int = howmuch.astype(int)
howmuch_int.to_csv(f'{workingdir}/colorado/co howmuch {year}.csv')

Total spills:
Number of Spills      0
Total Spill Volume    0
Name: Total, dtype: int64


In [ ]:
#@title Zip the full directory
# zip it!
import shutil
import zipfile

def zipdir(path, ziph):
    # ziph is zipfile handle
    for root, dirs, files in os.walk(path):
        for file in files:
          try: #ignores errors if it fails zipping a .gsheet
            ziph.write(os.path.join(root, file))
          except:
            continue

if __name__ == '__main__':
    directory_to_zip = f'{workingdir}/colorado/'
    output_file = f'{workingdir}/colorado.zip'

    zipf = zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED)
    zipdir(directory_to_zip, zipf)
    zipf.close()

# New Mexico
The first time you run the spills report, expand the `Import and setup` section. Expand the `Data prep` section and update the first cell. You may need to edit and run it a few times to make sure the list of operators doesn't contain duplicates.

## Import and setup

In [33]:
#@title Download spills data going back to 2022 if the box up top is checked
if nmdownload:
  url = (f'https://wwwapps.emnrd.nm.gov/OCD/OCDPermitting/Data/Spills/SpillSearchResultsExcel.aspx?IncidentIdSearchClause=BeginsWith&Severity=All&OperatorSearchClause=BeginsWith&FacilityIdSearchClause=BeginsWith&FacilityNameSearchClause=BeginsWith&WellNameSearchClause=BeginsWith&Incident_DateRangeStart=01/01/2022&Incident_DateRangeEnd=12/31/{year}&Section=00')
  try:
    print("Downloading New Mexico spills data...")
    request.urlretrieve(url, f'{workingdir}/{nmfilename}')
    print("Download successful.")
  except:
    print("Download failed, try downloading the data manually from the New Mexico website. Download five years' worth. Add to Google Drive folder.")

Download failed, try downloading the data manually from the New Mexico website. Download five years' worth. Add to Google Drive folder.


In [ ]:
#@title Bring the data into Pandas
file = (f'{workingdir}/{nmfilename}')

# Importing HTML into pandas creates a list of dataframes
tables = pd.read_html(file)

# The first list in each file is the dataframe we're working with
spillsall = tables[0]

# Rename these columns for easier coding
spillsall = spillsall.rename(columns={"Incident Number": "id",
                                "Incident Date": "date",
                                "Volume Released": "volume",
                                "Unit Of Volume": "unit"
                                })

# standardize capitalization of the units
spillsall['unit'] = spillsall['unit'].str.upper()

# Make the date column into datetime
spillsall['date'] = pd.to_datetime(spillsall['date'])

# Extract the Year, Month, Day to their own columns
spillsall['year'] = spillsall['date'].dt.year
spillsall['month'] = spillsall['date'].dt.month
spillsall['day'] = spillsall['date'].dt.day

# Make a Quarter column as well
def quarter(month):
  if month in (1,2,3):
    return 1
  elif month in (4,5,6):
    return 2
  elif month in (7,8,9):
    return 3
  elif month in (10,11,12):
    return 4
  else:
    return None

spillsall['quarter'] = spillsall['month'].apply(quarter)

# Fill in missing lease types
spillsall['Lease Type'] = spillsall['Lease Type'].fillna('Unknown')

### Helper functions

In [ ]:
# For sorting materials into two buckets, "Gas" and "Liquid"
def materialtype(material, incident):
  # List of gas material type
  gases = ["Natural Gas Vented", "Natural Gas Flared", "[OBSOLETE] Natural Gas (Methane)"]
  gasincidents = ["Flare", "Vent with Flaring",  "Natural Gas Release", "Vent"]
  if material in gases:
    return "Gas"
  # Liquid: All that aren't gas *and* aren't Other (Specify)
  elif material != "Other (Specify)":
    return "Liquid"
  ### For Other (Specify): If "Incident Type" is "Flare", "Vent with Flaring",
  ### "Natural Gas Release", or "Vent", materialtype is "Gas"
  elif material == "Other (Specify)":
    if incident in gasincidents:
      return "Gas"
    ### All other Other (Specify), materialtype is "Liquid"
    else:
      return "Liquid"
  # If this is something else entirely, leave it blank
  else:
    return None

# For making a standardized liquid volume column
def convert_liquid_volume(row):
    # convert barrels to gallons
    if row['unit'] == 'BBL':
        return row['volume'] * 42
    # Keep gallons as gallons
    elif row['unit'] == 'GAL':
        return row['volume']
    else:
        return None

# For making a standardized gas volume column
def convert_gas_volume(row):
  if row['unit'] == 'MCF':
    return row['volume']
  else:
    return None

In [ ]:
# For cleaning up county names
def remove_parens(text):
    # use regular expressions to match the pattern of a word or words followed by a number in parentheses
    pattern = re.compile(r'^(.+) \(\d+\)$')
    match = pattern.match(text)

    if match:
        # if the pattern is matched, return the matched group of words without the number and parentheses
        return match.group(1)
    else:
        # otherwise, return the original text
        return text

### Data prep — check and update the first cell!

In [ ]:
#@title Clean up operator names — check and update this cell!

# Make them all upper case
spillsall['Operator Name'] = spillsall['Operator Name'].str.upper()

print('Cleaning up operator names...')

# FRANKLIN MOUNTAIN
mask = spillsall['Operator Name'].str.startswith('FRANKLIN MOUNTAIN')
spillsall.loc[mask, 'Operator Name'] = 'FRANKLIN MOUNTAIN'

# MARALEX
mask = spillsall['Operator Name'].str.startswith('MARALEX')
spillsall.loc[mask, 'Operator Name'] = 'MARALEX'

# TARGA
mask = spillsall['Operator Name'].str.startswith('TARGA')
spillsall.loc[mask, 'Operator Name'] = 'TARGA'

# WESTERN REFINING
mask = spillsall['Operator Name'].str.startswith('WESTERN REFINING')
spillsall.loc[mask, 'Operator Name'] = 'WESTERN REFINING'

# WHIPTAIL
mask = spillsall['Operator Name'].str.startswith('WHIPTAIL')
spillsall.loc[mask, 'Operator Name'] = 'WHIPTAIL'

# CIMAREX
mask = spillsall['Operator Name'].str.startswith('CIMAREX')
spillsall.loc[mask, 'Operator Name'] = 'CIMAREX'

# COG
mask = spillsall['Operator Name'].str.startswith('COG')
spillsall.loc[mask, 'Operator Name'] = 'COG'

# CONTANGO
mask = spillsall['Operator Name'].str.startswith('CONTANGO')
spillsall.loc[mask, 'Operator Name'] = 'CONTANGO'

# DKL
mask = spillsall['Operator Name'].str.startswith('DKL')
spillsall.loc[mask, 'Operator Name'] = 'DKL'

# ETC TEXAS
mask = spillsall['Operator Name'].str.startswith('ETC TEXAS')
spillsall.loc[mask, 'Operator Name'] = 'ETC TEXAS'

# OXY
mask = spillsall['Operator Name'].str.startswith('OXY')
spillsall.loc[mask, 'Operator Name'] = 'OCCIDENTAL'

# OCCIDENTAL
mask = spillsall['Operator Name'].str.startswith('OCCIDENTAL')
spillsall.loc[mask, 'Operator Name'] = 'OCCIDENTAL'

# XTO
mask = spillsall['Operator Name'].str.startswith('XTO')
spillsall.loc[mask, 'Operator Name'] = 'XTO'

# Make a pivot table of operator names and save it
pivot = pd.pivot_table(spillsall, index="Operator Name",
                      values="id",
                      aggfunc="nunique")
print(f"Saving list of operator names to {workingdir}/newmexico/operator_names.csv")
pivot.to_csv(f'{workingdir}/newmexico/operator_names.csv')

Cleaning up operator names...
Saving list of operator names to /content/data/newmexico/operator_names.csv


Now is a good time to check `/data/newmexico/operator_names.csv` and look for duplicates. If there are, add them to the cell above and run it again.

In [ ]:
#@title Individual field cleanups and sanity check

#Remove the number from the County field
spillsall['County'] = spillsall.apply(lambda x:
                                      remove_parens(x['County']), axis=1)

# Create a column 'materialtype' and sort every row into two buckets, gas and liquid
spillsall['materialtype'] = spillsall.apply(lambda x:
                                      materialtype(x['Material'],
                                      x['Incident Type']), axis=1)

# Normalize the liquid volumes into gallons in a 'volume_gallons' field
spillsall['volume_gallons'] = spillsall.apply(lambda row: convert_liquid_volume(row), axis=1)

# Put only MCF releases into a 'volume_mcfs' field
spillsall['volume_mcfs'] = spillsall.apply(lambda row: convert_gas_volume(row), axis=1)

# Convert mcfs into cubic feet
spillsall['volume_cubicfeet'] = spillsall.apply(lambda x: cubic(x['volume_mcfs']), axis=1)

# save all of this to a csv file for spot checking
spillsall.to_csv(f'{workingdir}/newmexico/allspills.csv')

# make a 'spills' dataframe that is just the current year. this is a fresh copy
# rather than a reference to the big dataframe
spills = spillsall[spillsall['date'].dt.year == year].copy()

#make a spills_last dataframe that is just the previous year. also a fresh copy
spills_last = spillsall[spillsall['date'].dt.year == lastyear].copy()

# reality check
print('Row count:\n')
print(f'This year: {spills.shape}')
print(f'Last year: {spills_last.shape}')
print(f'All: {spillsall.shape}')

Row count:

This year: (42202, 41)
Last year: (54614, 41)
All: (139538, 41)


In [ ]:
#@title Export a CSV for the map

# Output current year with just the fields we need for the map
# Tracking, ID, Operator, Lat, Long

mapfields = ['Operator Name', 'Facility Name', 'Incident Type',
             'Lease Type', 'date', 'County', 'Material', 'volume', 'unit',
             'Spill Cause', 'Waterway Affected', 'Ground Water Impact',
             'Ground Water Depth', 'Latitude', 'Longitude', 'id']

#Only output liquid spills
liquids = spills[spills['materialtype'] == "Liquid"]
df_map = liquids[mapfields]

df_map.to_csv(f'{workingdir}/newmexico/mapdata.csv', index=False)

print(f'Map data saved to {workingdir}/newmexico/mapdata.csv')

Map data saved to /content/data/newmexico/mapdata.csv


## Analysis and output


In [ ]:
#@title Make the output file
reportfile = (f'{workingdir}/newmexico/newmexico_report.txt')
with open(reportfile, "w") as file:
  string = (hr +
            "NEW MEXICO " + str(year) + " SPILLS REPORT\n" +
            hr + "\n\n")
  file.write(string)

print("File " + reportfile + " created.")

File /content/data/newmexico/newmexico_report.txt created.


In [ ]:
#@title Total spills and volumes

##### FROM HERE ON OUT, WE'RE LOOKING AT THE CURRENT YEAR ONLY ######

# Add up the total volumes released and spilled
# Need to confirm no need to do duplicate detection here
# This also converts MCFs to cubic feet!
# 1 Mcf = 1000 cubic feet
gas_sum = spills['volume_cubicfeet'].sum()
liquid_sum = spills['volume_gallons'].sum()

# Use a pivot table rather than a straight count here to eliminate duplicate IDs.
# That could happen when one ID has multiple types of gas or liquids released
pivot = pd.pivot_table(spills, index='materialtype', aggfunc='nunique')['id']
gas_count = pivot.loc['Gas']
liquid_count = pivot.loc['Liquid']

#make a list of tuples with the numbers we care about, total and per day
out = [('Name', 'Total', 'Per Day'),
       ('Cubic feet of gas released', gas_sum, perday(gas_sum)),
       ('Number of gas releases', gas_count, perday(gas_count)),
       ('Gallons of liquid spilled', liquid_sum, perday(liquid_sum)),
       ('Number of liquid spills', liquid_count, perday(liquid_count))
       ]

# save that summary list to a csv file
with open(f'{workingdir}/newmexico/summary.csv', "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(out)

# Make two csv files that will go straight into Infogram
# nm summary1 is total number of spills
# nm summary2 is daily averages -- this is created in next cell

out = [(comma.format(liquid_count), 'oil-related spills'),
       (comma.format(gas_count), 'incidents of venting and flaring')]
with open(f'{workingdir}/newmexico/nm summary1 {year}.csv', "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(out)

# now make that human readable text, print it
# todo: Refactor this as a template
out = hr + "SUMMARY\n" + hr

out += (comma.format(gas_sum) + " cubic feet of gas vented and flared\n" +
        comma.format(perday(gas_sum)) + " cubic feet per day\n\n")

out += (comma.format(liquid_sum) + " gallons of liquid spilled\n" +
        comma.format(perday(liquid_sum)) + " gallons per day\n\n")

out += (comma.format(gas_count) + " gas releases\n" +
        small.format(perday(gas_count)) +
        " releases per day\n\n")
out += (comma.format(liquid_count) + " liquid spills\n" +
        small.format(perday(liquid_count)) +
        " spills per day\n\n")

print(out)

# add it to the report text file
with open(reportfile, "a") as file:
  file.write(out)

---------------
SUMMARY
---------------
11,474,286,000 cubic feet of gas vented and flared
31,436,400 cubic feet per day

4,810,923 gallons of liquid spilled
13,181 gallons per day

40,339 gas releases
110.5 releases per day

1,352 liquid spills
3.7 spills per day




In [ ]:
#@title Volume spilled/vented total/per day

# Group by Material
material_group = spills.groupby('Material')
gallons = material_group['volume_gallons'].sum().sort_values(ascending=False)
mcfs = material_group['volume_mcfs'].sum().sort_values(ascending=False)

# Take all of the liquids that aren't in the top three and combine them into an
# "Other" group
top_3 = gallons.iloc[:3]
others = gallons.iloc[3:].sum()
# Combine those series into a new one
gallons_others = top_3
gallons_others.loc['Other'] = others

# This is kludgy and a little risky -- assumes top 3 liquids are crude oil, produced water,
# and condensate. Right way would be to get these programattically
# Get the volumes we care about
gallons_oil = gallons['Crude Oil']
gallons_produced = gallons['Produced Water']
gallons_condensate = gallons['Condensate']
gallons_other = gallons_others['Other']
cubic_flared = cubic(mcfs['Natural Gas Flared'])
cubic_vented = cubic(mcfs['Natural Gas Vented'])

#make a list of tuples with the numbers we care about, total and per day
out = [('Name', 'Total', 'Per Day'),
       ('Crude oil spilled (gallons)', gallons_oil, perday(gallons_oil)),
       ('Produced water spilled (gallons)', gallons_produced, perday(gallons_produced)),
       ('Condensate spilled (gallons)', gallons_condensate, perday(gallons_condensate)),
       ('Other liquid spilled (gallons)', gallons_other, perday(gallons_other)),
       ('Natural gas flared (cubic feet)', cubic_flared, perday(cubic_flared)),
       ('Natural gas vented (cubic feet)', cubic_vented, perday(cubic_vented))
       ]

# Save this to a csv
with open(f'{workingdir}/newmexico/volumespilled.csv', "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(out)

# Create nm summary2 for Infogram -- first two rows are coming from the prior cell
out = [(comma.format(perday(liquid_count)), "spills per day"),
       (comma.format(perday(gas_count)), "venting and flaring events per day"),
       (comma.format(perday(gallons_oil)), "gallons of crude oil spilled per day"),
       (comma.format(perday(gallons_produced)), "gallons of produced water spilled per day"),
       (comma.format(perday(cubic_vented)), "cubic feet of methane vented per day"),
       (comma.format(perday(cubic_flared)), "cubic feet of methane flared per day")]

# Save this to a csv
with open(f'{workingdir}/newmexico/nm summary2 (per day) {year}.csv', "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(out)

# Create formatted 'what's in a spill' csv for Infogram
out = [('','volume'),
       ("Natural Gas Flared", f'{comma.format(cubic_flared / 1000)} thousand cubic feet'),
       ("Natural Gas Vented", f'{comma.format(cubic_vented / 1000)} thousand cubic feet'),
       ("Produced Water", f'{comma.format(gallons_produced)} gallons'),
       ("Crude Oil", f'{comma.format(gallons_oil)} gallons'),
       ("Other", f'{comma.format(gallons_other)} gallons'),
       ("Condensate", f'{comma.format(gallons_condensate)} gallons')
      ]
# Save this to a csv
with open(f'{workingdir}/newmexico/nm whats in a spill {year}.csv', "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(out)

# Todo: refactor this with a function/template.
out = hr + "Volume Spilled/Released\n" + hr
out += ("Crude oil spilled (gallons):\n" + comma.format(gallons_oil) + " total\n" +
        comma.format(perday(gallons_oil)) + " per day\n\n")
out += ("Produced water spilled (gallons):\n" + comma.format(gallons_produced) + " total\n" +
        comma.format(perday(gallons_produced)) + " per day\n\n")
out += ("Condensate spilled (gallons):\n" + comma.format(gallons_condensate) + " total\n" +
        comma.format(perday(gallons_condensate)) + " per day\n\n")
out += ("Other liquid spilled (gallons):\n" + comma.format(gallons_other) + " total\n" +
        comma.format(perday(gallons_other)) + " per day\n\n")
out += ("Natural gas flared (cubic feet):\n" + comma.format(cubic_flared) + " total\n" +
        comma.format(perday(cubic_flared)) + " per day\n\n")
out += ("Natural gas vented (cubic feet):\n" + comma.format(cubic_vented) + " total\n" +
        comma.format(perday(cubic_vented)) + " per day\n\n")

print (out)

# add it to the report text file
with open(reportfile, "a") as file:
  file.write(out)



---------------
Volume Spilled/Released
---------------
Crude oil spilled (gallons):
576,618 total
1,580 per day

Produced water spilled (gallons):
4,107,642 total
11,254 per day

Condensate spilled (gallons):
41,454 total
114 per day

Other liquid spilled (gallons):
79,321 total
217 per day

Natural gas flared (cubic feet):
10,806,146,000 total
29,605,879 per day

Natural gas vented (cubic feet):
595,916,000 total
1,632,647 per day




In [ ]:
#@title Trend since 2022
# Number of spills


In [ ]:
#@title Causes of methane releases

# Group by cause
gasspills = spills[spills['materialtype'] == 'Gas']
causes = gasspills.groupby('Spill Cause')['id'].count().sort_values(ascending=False)

# Take the top 5 that aren't "other". Combine the "other" and everything else
# into a final catch-all bucket
top_5_index = []

# Get the top 5 that aren't "Other"
for name in causes.index:
    if name != "Other":
      top_5_index.append(name)
    if len(top_5_index) >= 5:
      break

# Add up the rest, call it the new "Other"
other = 0
for cause in causes.index:
  if cause not in top_5_index:
    other += causes[cause]

# Build our new series with the top 5, add other
causes_others = pd.Series([], index=[], dtype="float64")
for cause in top_5_index:
  causes_others.loc[cause] = causes[cause]
causes_others.loc["Other"] = other

# Save that to a csv
causes_others.to_csv(f'{workingdir}/newmexico/causes_methane.csv')

# Formatted version of CSV for Infogram
def add_incident(value):
    return comma.format(value) + " incidents"

causes_others_with_incident = causes_others.apply(add_incident)
causes_others_with_incident.to_csv(f'{workingdir}/newmexico/nm causes methane {year}.csv')


# Add it to our report text file and print
out = (hr + "Causes of methane venting & flaring\n" + hr)
# take off the last "dtype: int64" from the string
# representation of the series
lines = str(causes_others).split('\n')[:-1]
clean = '\n'.join(lines)
out += (clean + "\n\n")

with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
Causes of methane venting & flaring
---------------
Equipment Failure                  11364
High Line Pressure                 10400
Repair and Maintenance              4387
Midstream Emergency Maintenance     3963
Normal Operations                   3798
Other                               5942




In [ ]:
#@title Causes of spills

# Group by cause
liquidspills = spills[spills['materialtype'] == 'Liquid']
causes = liquidspills.groupby('Spill Cause')['id'].count().sort_values(ascending=False)

# Take the top 5 that aren't "other". Combine the "other" and everything else
# into a final catch-all bucket
top_5_index = []

# Get the top 5 that aren't "Other"
for name in causes.index:
    if name != "Other":
      top_5_index.append(name)
    if len(top_5_index) >= 5:
      break

# Add up the rest, call it the new "Other"
other = 0
for cause in causes.index:
  if cause not in top_5_index:
    other += causes[cause]

# Build our new series with the top 5, add other
causes_others = pd.Series([], index=[], dtype="float64")
for cause in top_5_index:
  causes_others.loc[cause] = causes[cause]
causes_others.loc["Other"] = other

# Save that to a csv
causes_others.to_csv(f'{workingdir}/newmexico/causes_spills.csv')

# Formatted version of CSV for Infogram
def add_spills(value):
    return comma.format(value) + " spills"

causes_others_with_spills = causes_others.apply(add_spills)
causes_others_with_spills.to_csv(f'{workingdir}/newmexico/nm causes_spills {year}.csv')

# Add it to our report text file and print
out = (hr + "Causes of liquid spills\n" + hr)
# take off the last "dtype: int64" from the string
# representation of the series
lines = str(causes_others).split('\n')[:-1]
clean = '\n'.join(lines)
out += (clean + "\n\n")

with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
Causes of liquid spills
---------------
Equipment Failure    733
Corrosion            317
Human Error           94
Normal Operations     57
Fire                  54
Other                331




In [ ]:
#@title Sources of spills

#Now the same thing for sources of spills
sources = spills.groupby('Spill Source')['id'].count().sort_values(ascending=False)

# Take the top 5 that aren't "other". Combine the "other" and everything else
# into a final catch-all bucket
top_5_index = []

# Get the top 5 that aren't "Other (Specify)"
for name in sources.index:
    if name != "Other (Specify)":
      top_5_index.append(name)
    if len(top_5_index) >= 5:
      break

# Add up the rest, call it the new "Other"
other = 0
for source in sources.index:
  if source not in top_5_index:
    other += sources[source]

# Build our new series with the top 5, add other
sources_others = pd.Series([], index=[], dtype="float64")
for source in top_5_index:
  sources_others.loc[source] = sources[source]
sources_others.loc["Other"] = other

# Save that to a csv
sources_others.to_csv(f'{workingdir}/newmexico/sources.csv')

# Add it to our report text file and print
out = (hr + "Sources of spills\n" + hr)
# take off the last "dtype: int64" from the string
# representation of the series
lines = str(sources_others).split('\n')[:-1]
clean = '\n'.join(lines)
out += (clean + "\n\n")

with open(reportfile, "a") as file:
  file.write(out)

print(out)


---------------
Sources of spills
---------------
Producing Well            10056
Tank (Any)                 5966
Pipeline (Any)             4987
Gas Compressor Station     4268
Separator                  3084
Other                     13079




In [ ]:
#@title Producers with most liquid spilled

#group by producer, sum the gallons
producers_liquid = spills.groupby("Operator Name")["volume_gallons"].sum().sort_values(ascending=False)

#save that to a csv
producers_liquid.to_csv(f'{workingdir}/newmexico/producers_liquid.csv')

# get the top 5, add them up
top = producers_liquid[0:5]
top_5_liquid = top.sum()
# as a percentage of total volume of liquid spilled
total_liquid = spills["volume_gallons"].sum()
top_5_pct = top_5_liquid / total_liquid
# add up all the spills not in the top 5, add that to the Series
rest = producers_liquid[5:].sum()
top["OTHER"] = rest

# save that to a separate CSV
top.to_csv(f'{workingdir}/newmexico/producers_liquid_top.csv')

# add the word 'gallons' for Infogram
def add_gallons(value):
  return comma.format(value) + " gallons"

top_gallons = top.apply(add_gallons)
top_gallons.to_csv(f'{workingdir}/newmexico/nm producers_liquid_top {year}.csv')

# clean up the string representation
lines = str(top).split('\n')[1:-1]
clean = '\n'.join(lines)

out = (hr + "Producers with the most liquid spilled\n" +
       "(gallons)\n" + hr)
out += (clean + "\n\n")
out += (f"Five companies were responsible for {comma.format(top_5_pct * 100)} percent\nof liquid oil-related spills in {year}.\n\n")

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
Producers with the most liquid spilled
(gallons)
---------------
OCCIDENTAL                              506,730
LARIO OIL & GAS CO                      495,600
DEVON ENERGY PRODUCTION COMPANY, LP     350,490
XTO                                     336,126
MEWBOURNE OIL CO                        264,726
OTHER                                 2,857,251

Five companies were responsible for 41 percent
of liquid oil-related spills in 2024.




In [ ]:
#@title Year-to-year comparison for the top 5 liquid spillers

#Liquid only, year-to-year comparison
#group by producer, sum the gallons
producers_liquid_lastyear = spills_last.groupby("Operator Name")["volume_gallons"].sum().sort_values(ascending=False)

# merge with the top 5 dataframe
merged = pd.merge(top, producers_liquid_lastyear, on='Operator Name', how='inner')

# Calculate the difference between years
merged = merged.rename(columns = {'volume_gallons_x': f'volume_{year}', 'volume_gallons_y': f'volume_{lastyear}'})
merged['Difference'] = merged[f'volume_{year}'] - merged[f'volume_{lastyear}']
merged['Pct_Change'] = merged['Difference'] / merged[f'volume_{lastyear}']

# Save it to a CSV
merged.to_csv(f'{workingdir}/newmexico/producers_liquid_comparison.csv')

out = (f'{hr}Top 5 liquid spillers\nYear-to-year comparison{hr}')
out += (f'{merged}\n')

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
Top 5 liquid spillers
Year-to-year comparison---------------
                                     volume_2024  volume_2023  Difference  \
Operator Name                                                               
OCCIDENTAL                               506,730      175,140     331,590   
DEVON ENERGY PRODUCTION COMPANY, LP      350,490      407,190     -56,700   
XTO                                      336,126      238,266      97,860   
MEWBOURNE OIL CO                         264,726      199,038      65,688   

                                     Pct_Change  
Operator Name                                    
OCCIDENTAL                                    2  
DEVON ENERGY PRODUCTION COMPANY, LP          -0  
XTO                                           0  
MEWBOURNE OIL CO                              0  



In [ ]:
#@title Producers with most methane waste

# How many companies to include here?
n = 5

#group by producer, sum the gallons
producers_methane = spills.groupby("Operator Name")["volume_cubicfeet"].sum().sort_values(ascending=False)

#save that to a csv
producers_methane.to_csv(f'{workingdir}/newmexico/producers_methane.csv')

# get the top n, add them up
top = producers_methane[0:n]
top_n_methane = top.sum()
# as a percentage of total volume of methane leaked
total_methane = spills["volume_cubicfeet"].sum()
top_n_pct = top_n_methane / total_methane
# add up all the spills not in the top n, add that to the Series
rest = producers_methane[n:].sum()
top["OTHER"] = rest

# save that to a separate CSV
top.to_csv(f'{workingdir}/newmexico/producers_methane_top.csv')

# Add "cubic feet" for Infogram
def add_cubic(value):
  return comma.format(value / 1000) + " thousand cubic feet"

top_cubic = top.apply(add_cubic)
top_cubic.to_csv(f'{workingdir}/newmexico/nm producers_methane_top {year}.csv')

# clean up the string representation
lines = str(top).split('\n')[1:-1]
clean = '\n'.join(lines)

out = (hr + "Producers with the most methane waste\n" +
       "(cubic feet)\n" + hr)
out += (clean + "\n\n")
out += (f"These {str(n)} companies were responsible for {comma.format(top_n_pct * 100)} percent\nof New Mexico's methane waste in {year}.\n\n")

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
Producers with the most methane waste
(cubic feet)
---------------
EOG RESOURCES INC                  1,594,282,000
PERMIAN RESOURCES OPERATING, LLC   1,009,904,000
XTO                                  971,290,000
MATADOR PRODUCTION COMPANY           876,435,000
AVANT OPERATING, LLC                 850,645,000
OTHER                              6,171,730,000

These 5 companies were responsible for 46 percent
of New Mexico's methane waste in 2024.




In [ ]:
#@title Monthly methane emissions, top 10 producers

# Graph those producers' emissions over time, starting in July 2021
pivot = pd.pivot_table(spillsall,
                       index = ['Operator Name'],
                       columns = ['year', 'month'],
                       values = "volume_mcfs",
                       aggfunc = 'sum'
                       )
#sort it in descending order
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
pivot.to_csv(f'{workingdir}/newmexico/producers_methane_monthly.csv')
pivot.head(10).to_csv(f'{workingdir}/newmexico/producers_methane_monthly_top10.csv')

out = (f'Month-by-month methane emissions saved to {workingdir}/newmexico/producers_methane_monthly_top10.csv\n')

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)

print(out)


Month-by-month methane emissions saved to /content/data/newmexico/producers_methane_monthly_top10.csv



In [ ]:
#@title Number of spills by county

# A list of counties for making a CSV that turns into a map easily

counties = ['Bernalillo', 'Catron', 'Chaves', 'Cibola', 'Colfax', 'Curry',
            'De Baca', 'Doña Ana', 'Eddy', 'Grant', 'Guadalupe', 'Harding',
            'Hidalgo', 'Lea', 'Lincoln', 'Los Alamos', 'Luna', 'McKinley',
            'Mora', 'Otero', 'Quay', 'Rio Arriba', 'Roosevelt', 'Sandoval',
            'San Juan', 'San Miguel', 'Santa Fe', 'Sierra', 'Socorro', 'Taos',
            'Torrance', 'Union', 'Valencia']
# Matching list of coordinates for Infogram
county_coordinates = ["35.0512 -106.6707", "33.9155 -108.4049", "33.3626 -104.4669", "34.9636 -108.0564", "36.6064 -104.6461", "34.5742 -103.3469", "34.3425 -104.4119", "32.3526 -106.8328", "32.4828 -104.365", "32.7395 -108.3817", "34.863 -104.791", "35.858 -103.8194", "31.9146 -108.7144", "32.7854 -103.4338", "33.7452 -105.4566", "35.8639 -106.309", "32.1824 -107.7492", "35.58 -108.262", "36.0082 -104.9582", "32.6131 -105.7417", "35.1041 -103.55", "36.4653 -106.5772", "34.0214 -103.4807", "36.5002 -108.2334", "35.4798 -104.8164", "35.6885 -106.866", "35.5212 -105.9815", "33.1315 -107.1923", "34.0277 -107.1734", "36.4421 -105.6292", "34.6404 -105.8509", "36.4822 -103.4713", "34.7162 -106.8092"]

#make a dataframe with our county list
#county_df = pd.DataFrame(data=counties, columns=['County'])
#county_df = pd.DataFrame(data=(counties), index=range(0,len(counties)), columns=['County', 'Coordinates'])
county_df = pd.DataFrame({
    'County': counties,
    'Coordinates': county_coordinates,
    'Group': '',
    'Label': counties
})

# Only liquid spills
liquid = spills[spills['materialtype'] == 'Liquid']

# Make a pivot table by county -- counting spills
pivot = pd.pivot_table(liquid,
                       index = 'County',
                       values = 'id',
                       aggfunc = 'nunique'
                       ).sort_values(by='id', ascending=False)
# Merge that with the full county list
county_spillcount = county_df.copy()
merged = county_spillcount.merge(pivot, right_on='County', left_on='County', how='outer')
merged = merged.fillna(value=0)

# Reorder columns for infogram, convert to integer
merged = merged.reindex(columns=['County', 'id', 'Group', 'Coordinates', 'Label'])
merged['id'] = merged['id'].astype(int)
# If no spills, make it '' instead of 0
merged['id'] = merged['id'].apply(lambda x: '' if x == 0 else x)

# Save it
merged.to_csv(f'{workingdir}/newmexico/nm county_spills {year}.csv', index=False, header=False, encoding='utf-8')

# Print it
top_2 = pivot.sum(axis=1).nlargest(2).sum()
top_2_names = pivot.sum(axis=1).nlargest(2).index.tolist()
total = pivot['id'].sum()

lines = str(pivot).split('\n')[2:-1]
clean = '\n'.join(lines)

out = (hr + 'Liquid spills by county\n' + hr)
out += (clean + '\n\n')
out += ('The two largest counties, ' + top_2_names[0] + ' and ' + top_2_names[1] +
      ", accounted for\n" + comma.format(top_2) + " spills, or " +
      small.format(top_2 / total * 100) + " percent of all spills.\n\n")

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
Liquid spills by county
---------------
Lea         639
Eddy        584
San Juan     71
Rio Arriba   28
Sandoval     10
Harding       6
Chaves        6
Union         3
Roosevelt     3
McKinley      1

The two largest counties, Lea and Eddy, accounted for
1,223 spills, or 90.5 percent of all spills.




In [ ]:
#@title Volume spilled by county

# Only liquid spills
liquid = spills[spills['materialtype'] == 'Liquid']

# Make a pivot table by county -- adding up volume
pivot = pd.pivot_table(liquid,
                       index = 'County',
                       values = 'volume_gallons',
                       aggfunc = 'sum'
                       ).sort_values(by='volume_gallons', ascending=False)

pivot.to_csv(f'{workingdir}/newmexico/countyspills.csv')

# Print it and save it
top_2 = pivot.sum(axis=1).nlargest(2).sum()
top_2_names = pivot.sum(axis=1).nlargest(2).index.tolist()
total = pivot['volume_gallons'].sum()

lines = str(pivot).split('\n')[2:-1]
clean = '\n'.join(lines)

out = (hr + 'Gallons spilled by county\n' + hr)
out += (clean + '\n\n')
out += ('The two largest counties, ' + top_2_names[0] + ' and ' + top_2_names[1] +
      ", accounted for\n" + comma.format(top_2) + " gallons spilled, or " +
      small.format(top_2 / total * 100) + " percent of all liquid spilled.\n\n")

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)

print(out)

---------------
Gallons spilled by county
---------------
Lea              2,723,070
Eddy             1,834,350
San Juan           153,624
Sandoval            31,542
Rio Arriba          28,689
Chaves              18,732
McKinley            11,760
Roosevelt            4,284
Harding              2,814
Union                1,176

The two largest counties, Lea and Eddy, accounted for
4,557,420 gallons spilled, or 94.7 percent of all liquid spilled.




In [ ]:
#@title Number of methane waste reports by county

# Only gas releases
methane = spills[spills['materialtype'] == 'Gas']

# Make a pivot table by county -- counting releases
pivot = pd.pivot_table(methane,
                       index = 'County',
                       values = 'id',
                       aggfunc = 'nunique'
                       ).sort_values(by='id', ascending=False)
pivot.style.format('{:,}')

# Merge that with the full county list
county_releasecount = county_df.copy()  # county_df was created in the spills by county cell above
merged = county_releasecount.merge(pivot, right_on='County', left_on='County', how='outer')
merged = merged.fillna(value=0)

# Reorder columns for infogram, convert to integer
merged = merged.reindex(columns=['County', 'id', 'Group', 'Coordinates', 'Label'])
merged['id'] = merged['id'].astype(int)
# If no spills, make it '' instead of 0
merged['id'] = merged['id'].apply(lambda x: '' if x == 0 else x)

# Save it
merged.to_csv(f'{workingdir}/newmexico/nm county_ventflare {year}.csv', index=False, header=False, encoding='utf-8')


# Print it and save it
top_2 = pivot.sum(axis=1).nlargest(2).sum()
top_2_names = pivot.sum(axis=1).nlargest(2).index.tolist()
total = pivot['id'].sum()

lines = str(pivot).split('\n')[2:-1]
clean = '\n'.join(lines)

out = (hr + 'Venting/flaring reports by county\n' + hr)
out += (clean + '\n\n')
out += ('The two largest counties, ' + top_2_names[0] + ' and ' + top_2_names[1] +
      ", accounted for\n" + comma.format(top_2) +
      " methane venting/flaring reports, or " +
      small.format(top_2 / total * 100) + " percent\nof all reports.\n\n")

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)
print(out)


---------------
Venting/flaring reports by county
---------------
Lea         27055
Eddy        12478
Chaves        314
San Juan      187
Sandoval      160
Rio Arriba    137
Colfax          3
Union           3
McKinley        1

The two largest counties, Lea and Eddy, accounted for
39,533 methane venting/flaring reports, or 98.0 percent
of all reports.




In [ ]:
#@title Volume of methane released by county

# Only gas releases
methane = spills[spills['materialtype'] == 'Gas']

# Make a pivot table by county -- sum of releases
pivot = pd.pivot_table(methane,
                       index = 'County',
                       values = 'volume_cubicfeet',
                       aggfunc = 'sum'
                       ).sort_values(by='volume_cubicfeet', ascending=False)

pivot.to_csv(f'{workingdir}/newmexico/countymethane.csv')

# Print it and save it
top_2 = pivot.sum(axis=1).nlargest(2).sum()
top_2_names = pivot.sum(axis=1).nlargest(2).index.tolist()
total = pivot['volume_cubicfeet'].sum()

lines = str(pivot).split('\n')[2:-1]
clean = '\n'.join(lines)

out = (hr + 'Venting/flaring amounts by county\n' + hr)
out += (clean + '\n\n')
out += ('The two largest counties, ' + top_2_names[0] + ' and ' + top_2_names[1] +
      ", accounted for\n" + comma.format(top_2) + " cubic feet" +
      " of methane vented and flared,\nor " +
      small.format(top_2 / total * 100) + " percent of all venting and flaring.\n\n")

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)
print(out)


---------------
Venting/flaring amounts by county
---------------
Lea            7,032,350,000
Eddy           4,215,628,000
Chaves            45,465,000
Sandoval          41,576,000
San Juan          40,677,000
Rio Arriba        23,698,000
Union              1,850,000
Colfax               756,000
Roosevelt             62,000

The two largest counties, Lea and Eddy, accounted for
11,247,978,000 cubic feet of methane vented and flared,
or 98.6 percent of all venting and flaring.




In [ ]:
#@title Methane emissions year-to-year comparison

# This has some exception code that's ignored after 2022, because data changed in June 2021

# Just compare the second half of each year in 2022, so we're comparing apples to apples
if year == 2022:
  secondhalf = spillsall[spillsall['month'] >= 7]
  pivot = pd.pivot_table(secondhalf,
                      index=['year'],
                      values='volume_cubicfeet',
                      aggfunc='sum')
else:
  pivot = pd.pivot_table(spillsall,
                      index=['year'],
                      values='volume_cubicfeet',
                      aggfunc='sum')

# Save pivot to a csv file
pivot.to_csv(f'{workingdir}/newmexico/methane_year_comparison.csv')

diff = pivot.loc[year] - pivot.loc[lastyear]
pct_increase = ((pivot.loc[year] - pivot.loc[lastyear]) / pivot.loc[lastyear]) * 100

out = hr + "Methane Vented/Flared\n(cubic feet)\n"
if year == 2022:
  out += "July-December only\n"
out += hr
lines = str(pivot).split('\n')[2:]
clean = '\n'.join(lines)
out += (clean)
out += ("\n\nYear-to-year change:\n")
out += (comma.format(float(diff)) + "\n")
out += (comma.format(int(pct_increase)) + " percent change\n\n")

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)
print(out)


---------------
Methane Vented/Flared
(cubic feet)
---------------
2022    16,013,449,000
2023    20,202,404,000
2024    11,474,286,000

Year-to-year change:
-8,728,118,000
-43 percent change




In [ ]:
#from pandas.tseries.offsets import Nano
#@title Year-to-year comparison by operator (volume and percentage increase/decrease) for all methane releases and all liquid spills.

if year == 2022:
  secondhalf = spillsall[spillsall['month'] >= 7]
  pivot = pd.pivot_table(secondhalf,
                      index=['Operator Name', 'year'],
                      values=['volume_cubicfeet', 'volume_gallons'],
                      aggfunc='sum')
else:
  pivot = pd.pivot_table(spillsall,
                      index=['Operator Name', 'year'],
                      values=['volume_cubicfeet', 'volume_gallons'],
                      aggfunc='sum')

# put it in a dataframe
df = pd.DataFrame(columns=[
    f'gasvolume_{year}',
    f'gasvolume_{lastyear}',
    f'liquidvolume_{year}',
    f'liquidvolume_{lastyear}'
    ])

# Get a list of all of the operators in the pivot table
operators = [t[0] for t in list(pivot.index)]
# For every operator, create 4 columns:
# gasvolume_year, gasvolume_lastyear, liquidvolume_year, liquidvolume_lastyear

# Take that pivot table, make a new dataframe with columns for volume
# of each type of spill by year
for o in operators:
  try:
    gasvolume_year = pivot.loc[o].loc[year]['volume_cubicfeet']
  except KeyError:
    gasvolume_year = math.nan
  try:
    gasvolume_lastyear = pivot.loc[o].loc[lastyear]['volume_cubicfeet']
  except KeyError:
    gasvolume_lastyear = math.nan
  try:
    liquidvolume_year = pivot.loc[o].loc[year]['volume_gallons']
  except KeyError:
    liquidvolume_year = math.nan
  try:
    liquidvolume_lastyear = pivot.loc[o].loc[lastyear]['volume_gallons']
  except KeyError:
    liquidvolume_lastyear = math.nan
  df.loc[o] = [gasvolume_year, gasvolume_lastyear,
               liquidvolume_year, liquidvolume_lastyear]

#Calculate year-to-year differences and percentage changes
df['gasvolume_difference'] = df[f'gasvolume_{year}'] - df[f'gasvolume_{lastyear}']
df['gasvolume_pct'] = df['gasvolume_difference'] / df[f'gasvolume_{lastyear}']
df['liquidvolume_difference'] = df[f'liquidvolume_{year}'] - df[f'liquidvolume_{lastyear}']
df['liquidvolume_pct'] = df['liquidvolume_difference'] / df[f'liquidvolume_{lastyear}']

# save it to a CSV
df.to_csv(f'{workingdir}/newmexico/producers_year_comparison.csv')

# get the top 5, methane only, save that to a csv
drop_columns = [f'liquidvolume_{year}', f'liquidvolume_{lastyear}', 'liquidvolume_difference', 'liquidvolume_pct']
top = df.sort_values(by=f'gasvolume_{year}', ascending=False).head(5).drop(columns=drop_columns)
top.to_csv(f'{workingdir}/newmexico/producers_methane_comparison.csv')

out = (f'{hr}Methane releases by company (top 5)\nYear-to-year\n{hr}')
out += (f'{top}\n\n')

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)
print(out)

---------------
Methane releases by company (top 5)
Year-to-year
---------------
                                  gasvolume_2024  gasvolume_2023  \
EOG RESOURCES INC                  1,594,282,000   2,154,566,000   
PERMIAN RESOURCES OPERATING, LLC   1,009,904,000   3,267,343,000   
XTO                                  971,290,000   1,496,371,000   
MATADOR PRODUCTION COMPANY           876,435,000     577,868,000   
AVANT OPERATING, LLC                 850,645,000   1,036,249,000   

                                  gasvolume_difference  gasvolume_pct  
EOG RESOURCES INC                         -560,284,000             -0  
PERMIAN RESOURCES OPERATING, LLC        -2,257,439,000             -1  
XTO                                       -525,081,000             -0  
MATADOR PRODUCTION COMPANY                 298,567,000              1  
AVANT OPERATING, LLC                      -185,604,000             -0  




In [ ]:
#@title Count venting and flaring incidents separately
if year == 2022:
  secondhalf = spillsall[spillsall['month'] >= 7]
  pivot = pd.pivot_table(secondhalf,
                      index=['year', 'Material'],
                      values='id',
                      aggfunc='nunique')
else:
  pivot = pd.pivot_table(spillsall,
                      index=['year', 'Material'],
                      values='id',
                      aggfunc='nunique')

#put it in a dataframe
df = pd.DataFrame()
df['Number of Incidents'] = ['Natural Gas Flared', 'Natural Gas Vented']
for y in [year, lastyear]:
  df[y] = [pivot.loc[y].loc['Natural Gas Flared']['id'], pivot.loc[y].loc['Natural Gas Vented']['id']]

#make a difference column
df['Difference'] = df[year] - df[lastyear]
df.set_index("Number of Incidents", inplace=True)

#save it to a csv
df.to_csv(f'{workingdir}/newmexico/vent_flare_count.csv')

# Pretty text string
out = hr + "Number of Venting & Flaring Incidents\n"
if year == 2022:
  out += "July-December only\n"
out += hr
lines = str(df).split('\n')
clean = '\n'.join(lines)
out += (clean)
out += '\n\n'

# Add it to the report text file, print it
with open(reportfile, "a") as file:
  file.write(out)
print(out)

---------------
Number of Venting & Flaring Incidents
---------------
                      2024   2023  Difference
Number of Incidents                          
Natural Gas Flared   38933  49866      -10933
Natural Gas Vented     917   1564        -647




## Zip it and output

In [ ]:
# zip it!

import shutil
import zipfile

def zipdir(path, ziph):
    # ziph is zipfile handle
    for root, dirs, files in os.walk(path):
        for file in files:
          try:
            ziph.write(os.path.join(root, file))
          except:
            continue

if __name__ == '__main__':
    directory_to_zip = (f'{workingdir}/newmexico/')
    output_file = (f'{workingdir}/newmexico.zip')

    zipf = zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED)
    zipdir(directory_to_zip, zipf)
    zipf.close()

# Wyoming
Expand this and read the `Read me first` section. Wyoming data is messy and requires extra attention to clean it up.

## Read me first!

Wyoming doesn't make spills data available for download on demand. You need to request data from the state. Email patrick.amole1@wyo.gov and ask for data (send last year’s file as example).

Name that spreadsheet `WyomingSpills[20xx].xls` and put it in your working directory for this year's spills. Put last year's spreadsheet in the same directory as well. If Pandas gives you an error reading the Excel sheet, export it to a CSV and use that instead.

The Wyoming data is messy, with inconsistent company names. Run this section then pay attention to the part where you should check the list of operators and add extra find/replace terms in these cells if necessary.

In [ ]:
#@title Import the files
#is this a csv or an Excel file?
if wyfilename1.endswith(".csv"):
  spills1 = pd.read_csv(f'{workingdir}/{wyfilename1}')
elif wyfilename1.endswith(".xls") or wyfilename1.endswith(".xlsx"):
  spills1 = pd.read_excel(f'{workingdir}/{wyfilename1}')
else:
  print("Filename 1 is not Excel or CSV.")

if wyfilename2.endswith(".csv"):
  spills2 = pd.read_csv(f'{workingdir}/{wyfilename2}')
elif wyfilename2.endswith(".xls") or wyfilename2.endswith(".xlsx"):
  spills2 = pd.read_excel(f'{workingdir}/{wyfilename2}')
else:
  print("Filename is not Excel or CSV.")

#Combine the two years
spillsall = pd.concat([spills1, spills2])

# emptying the spills variable just to make sure we don't reference it
# until we create a new one based on spillsall
spills = []

In [ ]:
#@title Clean up the data

#make the date fields into dates
datefields = ['REP_DATE', 'DIS_DATE', 'INC_DATE', 'DATE_CONTROLLED']
for field in datefields:
    spillsall[field] = pd.to_datetime(spillsall[field], errors='coerce')

# Function to convert barrels to gallons if necessary
def togallons(vol, unit):
  if unit == 'Bbls':
    return vol * 42
  elif unit == "Gal":
    return vol
  else:
    return None

# Which columns do we need to standardize?
standardize = ['DIS_OIL', 'DIS_WATER', 'DIS_OTHER']

# Create a standardized column for each one in the list
for col in standardize:
 spillsall[col] = spillsall[col].fillna(0)
 spillsall[f'{col}_GAL'] = spillsall.apply(lambda x: togallons(x[col], x['BBLS_GALS']), axis=1)

# Create a total gallons spilled column
spillsall['DIS_TOTAL_GAL'] = spillsall['DIS_OIL_GAL'] + spillsall['DIS_WATER_GAL'] + spillsall['DIS_OTHER_GAL']

#create a unique ID for each spill since the spreadsheet doesn't have one
spillsall = spillsall.reset_index()
spillsall = spillsall.rename(columns={'index': 'ID'})

#clean up the county names
spillsall['COUNTYNAME'] = spillsall['COUNTYNAME'].str.strip()


In [ ]:
#@title Standardize company names

# Make everything upper case
spillsall['PRO_NAME_CLEAN'] = spillsall['PRO_NAME'].str.upper()
names = pd.pivot_table(spillsall, index="PRO_NAME_CLEAN", values="ID", aggfunc="nunique")
print(f'Length after upper: {len(names)}')

# Get rid of punctuation
spillsall['PRO_NAME_CLEAN'] = spillsall['PRO_NAME_CLEAN'].str.replace(r'[,.]', '', regex=True).str.strip()
names = pd.pivot_table(spillsall, index="PRO_NAME_CLEAN", values="ID", aggfunc="nunique")
print(f'Length after punctuation: {len(names)}')

# Get rid of LLC, LP, INC, LTD, LLCSW, Corp, Operating,
# Company, Corporation, Corp., Production, Company, etc.
#@markdown List of terms to erase from the end of company names.
#@markdown Put each term in quotes `''` with a comma `,` between each term.
#remove = ['ENERGY', 'PETROLEUM', 'OIL AND GAS', 'CO', 'LLP', 'LP', 'INC', 'LTD', 'LLCSW', 'CORP', 'OPERATING', 'COMPANY', 'CORPORATION', 'CORP', 'PRODUCTION', 'COMPANY', 'LLC', 'HOLDINGS', 'ONSHORE', 'OIL \& GAS', 'INCSE', 'USA', 'EXPLORATION', 'RESOURCES', 'LCC', 'E\&P',  ' OIL',  'OPERATORS', '\&', 'DIVISION', 'OPERATORS']
remove = ['ENERGY', 'PETROLEUM', 'OIL & GAS', 'CO', 'LLP', 'LP', 'INC', 'LTD', 'LLCSW', 'CORP', 'OPERATING', 'COMPANY', 'CORPORATION', 'PRODUCTION', 'LLC', 'HOLDINGS', 'ONSHORE', 'INCSE', 'USA', 'EXPLORATION', 'RESOURCES', 'LCC', 'E&P', 'OIL', 'OPERATORS', 'DIVISION'] #@param {type:"raw"}

# Make a regex pattern and do the cleanup
#pattern = r'({})$'.format('|'.join(remove))
pattern = r"(\b" + r"\b|\b".join(remove) + r"\b)\s*$"  # Add \b for word boundaries, \s* to match trailing spaces
print(f'Regex pattern: {pattern}')
spillsall['PRO_NAME_CLEAN'] = spillsall['PRO_NAME_CLEAN'].str.replace(pattern, '').str.strip()
names = pd.pivot_table(spillsall, index="PRO_NAME_CLEAN", values="ID", aggfunc="nunique")
print(f'Length after first pass: {len(names)}')

# get a list of operator names. Run the cleanup again and see if they match.
# If they don't, run the cleanup again until they match.
print("Cleaning...")
names = pd.pivot_table(spillsall, index="PRO_NAME_CLEAN", values="ID", aggfunc="nunique")
names1len = len(names)
names2len = 0

# keep looping twice until both passes match
while names1len != names2len:
  print(f'Cleaning again...')
  print(f'names1: {names1len}')
  print(f'names2: {names2len}')
  spillsall['PRO_NAME_CLEAN'] = spillsall['PRO_NAME_CLEAN'].str.replace(pattern, '', regex=True).str.strip()
  names =  pd.pivot_table(spillsall, index="PRO_NAME_CLEAN", values="ID", aggfunc="nunique")
  names2len = len(names)
  print(f'names1: {names1len}')
  print(f'names2: {names2len}')
  if names1len == names2len:
    print('No more edits, exiting loop.')
    continue
  else:
    names1len = names2len
    names2len = 0
    print('No match, looping...')

# Save and print the list of operators so we can look for any remaining ones
# to clean up
print(f"Saving list of operator names to {workingdir}/wyoming/operator_names.csv")
names.to_csv(f'{workingdir}/wyoming/operator_names.csv')

Length after upper: 182
Length after punctuation: 163
Regex pattern: (\bENERGY\b|\bPETROLEUM\b|\bOIL & GAS\b|\bCO\b|\bLLP\b|\bLP\b|\bINC\b|\bLTD\b|\bLLCSW\b|\bCORP\b|\bOPERATING\b|\bCOMPANY\b|\bCORPORATION\b|\bPRODUCTION\b|\bLLC\b|\bHOLDINGS\b|\bONSHORE\b|\bINCSE\b|\bUSA\b|\bEXPLORATION\b|\bRESOURCES\b|\bLCC\b|\bE&P\b|\bOIL\b|\bOPERATORS\b|\bDIVISION\b)\s*$
Length after first pass: 163
Cleaning...
Cleaning again...
names1: 163
names2: 0
names1: 163
names2: 152
No match, looping...
Cleaning again...
names1: 152
names2: 0
names1: 152
names2: 137
No match, looping...
Cleaning again...
names1: 137
names2: 0
names1: 137
names2: 134
No match, looping...
Cleaning again...
names1: 134
names2: 0
names1: 134
names2: 133
No match, looping...
Cleaning again...
names1: 133
names2: 0
names1: 133
names2: 133
No more edits, exiting loop.
Saving list of operator names to /content/data/wyoming/operator_names.csv


In [ ]:
#@title Other operator names to clean up — check and update this cell!

# Edit and add to this cell to clean up any remaining operator names
# that weren't cleaned up by the previous cell.

# This includes inconsistent formatting, individual site names, etc.

# ATR
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('ATR')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'ATR'

# ATR
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('E&B')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'E&B'

# LEGACY IS NOW REVENIR
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('LEGACY')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'REVENIR'

# MARATHON
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('MARATHON')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'MARATHON'

# NORTH SHORE
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('NORTH SHORE')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'NORTH SHORE'

# PEAK POWDER RIVER
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('PEAK POWDER RIVER')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'PEAK POWDER RIVER'

# CITATION
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('CITATION')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'CITATION'

# SAGE BUTTE
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('SAGE BUTTE')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'SAGE BUTTE'

# ULTRA
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('ULTRA')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'ULTRA'

# SUNSHINE VALLEY
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('SUNSHINE VALLEY')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'SUNSHINE VALLEY'

# MARATHON
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('MARATHON')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'MARATHON'

# REVENIR
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('LEGACY RESERVES')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'REVENIR'

# E&B NATURAL
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('E&B NATURAL')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'E&B NATURAL'

# CONTANGO
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('CONTNAGO')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'CONTANGO'

# POC-I
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('POC - I')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'POC-I'

# CONOCO
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('CONOCOPHILLIPS')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'CONOCOPHILLIPS'

# BURLINGTON (TYPO)
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('BURINGTON')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'BURLINGTON'

# SAMSON
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('SAMSON')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'SAMSON'

# GREEN RESERVE
mask = spillsall['PRO_NAME_CLEAN'].str.startswith('GREEN RESERVE')
spillsall.loc[mask, 'PRO_NAME_CLEAN'] = 'GREEN RESERVE'

pivot = pd.pivot_table(spillsall, index="PRO_NAME_CLEAN",
                      values="ID",
                      aggfunc="nunique")
print(f"Saving list of operator names to {workingdir}/wyoming/operator_names.csv")
pivot.to_csv(f'{workingdir}/wyoming/operator_names.csv')

print(f"Saving all spills to {workingdir}/wyoming/allspills.csv")
spillsall.to_csv(f'{workingdir}/wyoming/allspills.csv')

Saving list of operator names to /content/data/wyoming/operator_names.csv
Saving all spills to /content/data/wyoming/allspills.csv


In [ ]:
# Make a dataframe for just the events reported in the current year,
# and one for the previous year
spills = spillsall[(spillsall['REP_DATE'] >= f'{year}-01-01') & (spillsall['REP_DATE'] <= f'{year}-12-31')]
spillslast = spillsall[(spillsall['REP_DATE'] >= f'{lastyear}-01-01') & (spillsall['REP_DATE'] <= f'{lastyear}-12-31')]
print(f'Number of {year} spills: {spills.shape[0]}')
print(f'Number of {lastyear} spills: {spillslast.shape[0]}')


Number of 2024 spills: 825
Number of 2023 spills: 730


## Manual check of operator names

Now is a good time to open up /data/wyoming/operator_names.csv from the file browser to the left. Look at all of the operator names. Are there any duplicates that need to be consolidated?

If so, edit the two lists in the cells above then run the Wyoming section again.

## Run the reports

In [ ]:
#@title Create the master report file
reportfile = (f'{workingdir}/wyoming/wyoming_report.txt')
with open(reportfile, "w") as file:
  string = (hr +
            "WYOMING " + str(year) + " SPILLS REPORT\n" +
            hr + "\n")
  file.write(string)

print("File " + reportfile + " created.")

File /content/data/wyoming/wyoming_report.txt created.


In [ ]:
#@title Number of spills by material type
oil = (spills['DIS_OIL_GAL'] > 0).sum()
water = (spills['DIS_WATER_GAL'] > 0).sum()
other = (spills['DIS_OTHER_GAL'] > 0).sum()
total = oil + water + other

# Make a dataframe, save it to csv
spillstype = {
    'Type': ['Oil', 'Produced Water', 'Other'],
    'Number of spills': [oil, water, other]
}
spillstype = pd.DataFrame(spillstype).sort_values(by='Number of spills', ascending=False)
spillstype.to_csv(f'{workingdir}/wyoming/wy material_types {year}.csv', index=False)

out = (f'{hr}Number of spills by type\n{hr}')
out += (f'Oil: {oil}\n')
out += (f'Produced water: {water}\n')
out += (f'Other: {other}\n')
out += (f'Total: {total}\n\n')

# Make a nice string, save and print it
with open(reportfile, 'a') as file:
  file.write(out)

print (out)

---------------
Number of spills by type
---------------
Oil: 350
Produced water: 558
Other: 64
Total: 972




In [ ]:
#@title Volume of spills by material type

# Summarize and calculate condensate spill volume

oil = spills['DIS_OIL_GAL'].sum()
water = spills['DIS_WATER_GAL'].sum()
other = spills['DIS_OTHER_GAL'].sum()
total = oil + water + other

# Make a dataframe, save it to csv
spillsvol = {
    'Type': ['Oil', 'Produced Water', 'Other'],
    'Volume spilled (gallons)': [oil, water, other]
}
spillsvol = pd.DataFrame(spillsvol).sort_values(by='Volume spilled (gallons)',
                                                ascending=False)
spillsvol['Volume spilled (gallons)'] = spillsvol['Volume spilled (gallons)'].astype(int)
spillsvol.to_csv(f'{workingdir}/wyoming/wy material_volume {year}.csv', index=False)

# Make a nice string, save and print it
out = (f'{hr}Volume spilled (gallons)\n{hr}')
out += (f'Oil: {comma.format(oil)}\n')
out += (f'Produced water: {comma.format(water)}\n')
out += (f'Other: {comma.format(other)}\n\n')
out += (f'Total: {comma.format(total)}\n\n')

with open(reportfile, 'a') as file:
  file.write(out)

print(out)

---------------
Volume spilled (gallons)
---------------
Oil: 190,520
Produced water: 1,524,543
Other: 93,934

Total: 1,808,997




In [ ]:
#@title County list
counties = ['Albany', 'Big Horn', 'Campbell', 'Carbon', 'Converse', 'Crook',
            'Fremont', 'Goshen', 'Hot Springs', 'Johnson', 'Laramie', 'Lincoln',
            'Natrona', 'Niobrara', 'Park', 'Platte', 'Sheridan', 'Sublette',
            'Sweetwater', 'Teton', 'Uinta', 'Washakie', 'Weston']
#counties = [s.upper() for s in counties]
county_coordinates = ["41.6684 -105.6813", "44.4953 -108.0025", "44.1908 -105.5775", "41.7423 -106.8626", "43.0202 -105.5046", "44.5747 -104.5816", "43.0293 -108.5773", "42.0657 -104.3541", "43.7009 -108.4096", "44.0156 -106.6141", "41.2737 -104.7268", "42.1766 -110.7208", "42.9487 -106.8466", "43.0088 -104.4566", "44.6076 -109.5678", "42.0923 -104.9871", "44.7948 -106.8223", "42.7369 -109.9906", "41.6239 -108.9702", "43.9139 -110.638", "41.2632 -110.5679", "43.9309 -107.6704", "43.8398 -104.5362"]
#counties_df = pd.DataFrame(data=[counties, coords], index=range(0,len(counties)), columns = ['COUNTYNAME', 'COORDS'])
counties_df = pd.DataFrame({
    'COUNTYNAME': counties,
    'Coordinates': county_coordinates,
    'Group': '',
    'Label': counties
})

In [ ]:
#@title Number of spills by county

# How many counties to include here?
n = 2

pivot = spills.pivot_table(index = 'COUNTYNAME', values = 'ID',
                           aggfunc='nunique').sort_values(by='ID', ascending=False)

# merge the tables
merged = counties_df.merge(pivot, left_on='COUNTYNAME', right_on='COUNTYNAME', how='outer')
merged = merged.fillna(value=0)

# Reorder columns for infogram, convert to integer
merged = merged.reindex(columns=['COUNTYNAME', 'ID', 'Group', 'Coordinates', 'Label'])
merged['ID'] = merged['ID'].astype(int)
# If no spills, make it '' instead of 0
merged['ID'] = merged['ID'].apply(lambda x: '' if x == 0 else x)

merged.to_csv(f'{workingdir}/wyoming/wy county {year}.csv', index=False, header=False)

total = pivot["ID"].sum()
# Make pretty text, add it to the report doc
out = (f'{hr}Spills by county\n{hr}')
lines = str(pivot).split('\n')[2:]
clean = '\n'.join(lines)
out += (f'{clean}\n\n')
out += (f'Total: {total}\n\n')

# Get the top n
top_n = pivot['ID'].iloc[:n]
top_n_sum = top_n.sum()
out += (f'The top {n} counties, {top_n.index[0]} and {top_n.index[1]}, accounted for\n')
out += (f'{top_n_sum} spills, or {small.format(top_n_sum / total * 100)} percent of all spills.')

with open(reportfile, 'a') as file:
  file.write(out)

print(out)

---------------
Spills by county
---------------
Converse     174
Campbell     144
Sweetwater   114
Carbon        73
Laramie       69
Sublette      40
Natrona       37
Unknown       33
Fremont       24
Park          21
Hot Springs   21
Johnson       20
Weston        14
Crook         11
Washakie       7
Uinta          7
Niobrara       6
Big Horn       4
Lincoln        3
Albany         2
Sheridan       1

Total: 825

The top 2 counties, Converse and Campbell, accounted for
318 spills, or 38.5 percent of all spills.


In [ ]:
#@title Number of spills by producer

# How many companies to include here?
n = 5

pivot = spills.pivot_table(index = 'PRO_NAME_CLEAN', values = 'ID',
                           aggfunc='nunique').sort_values(by='ID', ascending=False)
# save the CSV file
pivot.to_csv(f'{workingdir}/wyoming/wy producer_spills {year}.csv', index=False)

# Get the top n
top_n = pivot['ID'].iloc[:n]

#take the others that aren't in the top 10, make an "other" bin
others = pivot['ID'].iloc[n:]
others_sum = others.sum()
top_n['OTHERS'] = others_sum

# Save that to a CSV for easy graphs
top_n.to_csv(f'{workingdir}/wyoming/wy producer_spills_top {year}.csv', index=False)

# Make a pretty table
out = (f'{hr}Producers with the most spills\n{hr}')
lines = str(top_n).split('\n')[1:-1]
clean = '\n'.join(lines)
out += (f'{clean}\n\n')
out += (f'Total: {top_n.sum()}')

print(out)

with open(reportfile, 'a') as file:
  file.write(out)


---------------
Producers with the most spills
---------------
CROWHEART      145
CONTINENTAL     93
CONTANGO        63
EOG             59
MERIT           35
OTHERS         430

Total: 825


In [ ]:
#@title Volume of spills by producer

# How many companies to include here?
n = 5

pivot = spills.pivot_table(index = 'PRO_NAME_CLEAN', values = 'DIS_TOTAL_GAL',
                           aggfunc='sum').sort_values(by='DIS_TOTAL_GAL', ascending=False)
# save the CSV file
pivot.to_csv(f'{workingdir}/wyoming/producer_spills_volume.csv')

# Get the top n
top_n = pivot['DIS_TOTAL_GAL'].iloc[:n]
top_n_total =  top_n.sum()

#take the others that aren't in the top 10, make an "other" bin
others = pivot['DIS_TOTAL_GAL'].iloc[n:]
others_sum = others.sum()
top_n['OTHERS'] = others_sum
total = top_n.sum()

# Save that to a CSV for easy graphs
formatted_top_n = top_n.apply(lambda x: "{:,.0f} gallons".format(x))
formatted_top_n.to_csv(f'{workingdir}/wyoming/wy producer_spills_volume_top {year}.csv', index=True)

# Make a pretty table
out = (f'{hr}Producers with the most volume spilled (gallons)\n{hr}')
lines = str(top_n).split('\n')[1:-1]
clean = '\n'.join(lines)
out += (f'{clean}\n\n')
out += (f'Total: {comma.format(total)}\n\n')

out += (f'The top {n} producers accounted for\n')
out += (f'{comma.format(top_n_total)} gallons spilled, or\n{small.format(top_n_total / total * 100)} percent of all gallons spilled.')

print(out)

with open(reportfile, 'a') as file:
  file.write(out)

---------------
Producers with the most volume spilled (gallons)
---------------
CARBON CREEK   321,409
EOG            187,091
CONTINENTAL    143,559
CROWHEART      120,847
ANSCHUTZ       112,602
OTHERS         923,490

Total: 1,808,997

The top 5 producers accounted for
885,507 gallons spilled, or
49.0 percent of all gallons spilled.


In [ ]:
#@title Volume spilled by producer, year-to-year comparison

# How many companies to include here?
n = 5

# Make a pivot table for both years
pivotyear = spills.pivot_table(index = 'PRO_NAME_CLEAN', values = 'DIS_TOTAL_GAL',
                           aggfunc='sum').sort_values(by='DIS_TOTAL_GAL', ascending=False)

pivotlast = spillslast.pivot_table(index = 'PRO_NAME_CLEAN', values = 'DIS_TOTAL_GAL',
                           aggfunc='sum').sort_values(by='DIS_TOTAL_GAL', ascending=False)

# Get the top 5 from the current year
top = pivotyear.sort_values(by='DIS_TOTAL_GAL', ascending=False).head(5)

# Merge with the previous year
merged = pd.merge(top, pivotlast, on='PRO_NAME_CLEAN', how='left')
merged = merged.rename(columns={'DIS_TOTAL_GAL_x': f'DIS_TOTAL_GAL_{year}', 'DIS_TOTAL_GAL_y': f'DIS_TOTAL_GAL_{lastyear}'})

# Make comparison columns
merged['Difference'] = merged[f'DIS_TOTAL_GAL_{year}'] - merged[f'DIS_TOTAL_GAL_{lastyear}']
merged['Pct_Change'] = merged['Difference'] / merged[f'DIS_TOTAL_GAL_{lastyear}']

# convert to integers for Infogram
merged.fillna(0, inplace=True)
int_columns = [f'DIS_TOTAL_GAL_{year}', f'DIS_TOTAL_GAL_{lastyear}', 'Difference']
for c in int_columns:
  merged[c] = merged[c].astype(int)
merged[merged == 0] = "N/A"
merged.to_csv(f'{workingdir}/wyoming/wy producer_spills_year {year}.csv', index=True)
#Rename the column and index for Infogram
new_column_name = f"Percent Change in Volume Spilled (Compared to {lastyear})"
pct_only = merged.iloc[:,3]
pct_only = pct_only.rename(new_column_name)
pct_only = pct_only.rename_axis('Producer')
pct_only.to_csv(f'{workingdir}/wyoming/wy producer_spills_difference {year}.csv', index=True)

out = (f'{hr}Producers with the most volume spilled (gallons)\nYear-to-year comparison\n{hr}')
out += (f'{merged}\n')

print(out)

with open(reportfile, 'a') as file:
  file.write(out)

---------------
Producers with the most volume spilled (gallons)
Year-to-year comparison
---------------
                DIS_TOTAL_GAL_2024  DIS_TOTAL_GAL_2023  Difference  Pct_Change
PRO_NAME_CLEAN                                                                
CARBON CREEK                321408               28039      293369          10
EOG                         187090               85627      101463           1
CONTINENTAL                 143559               36218      107340           3
CROWHEART                   120846               82313       38533           0
ANSCHUTZ                    112602               57663       54938           1



## Output and download



In [ ]:
# zip it!

import shutil
import zipfile

def zipdir(path, ziph):
    # ziph is zipfile handle
    for root, dirs, files in os.walk(path):
        for file in files:
            ziph.write(os.path.join(root, file))

if __name__ == '__main__':
    directory_to_zip = f'{workingdir}/wyoming/'
    output_file = f'{workingdir}/wyoming.zip'

    zipf = zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED)
    zipdir(directory_to_zip, zipf)
    zipf.close()

# Production Data
You shouldn't need to do anything here in future years except run all the cells. Just hit the play button.

## Colorado

The Colorado data is at https://cogcc.state.co.us/data2.html#/downloads.

Note that Colorado has 4 different production datasets, all with different numbers.

We're using the county production data.

###Source: Production by county, generated from https://cogcc.state.co.us/data4.html#/production

In [ ]:
#@title Install mechanize for using the interactive form
!pip install mechanize
import mechanize

# List of tables for future reference:
# Oil Produced 0
# Oil Sold 1
# Natural and Coalbed Gas Produced 2
# Natural and Coalbed Gas Sold 3
# Carbon Dioxide (CO2) Produced 4
# Carbon Dioxide (CO2) Sold 5
# Coalbed Methane Produced 6
# Coalbed Methane Sold 7
# Water Produced 8


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.4/110.4 kB 2.1 MB/s eta 0:00:00


In [ ]:
#@title Function to get Colorado production data from https://cogcc.state.co.us/production/ReportPortal

def getproduction (year):
  # create a browser object using mechanize
  br = mechanize.Browser()
  br.set_handle_robots(False)   # ignore robots
  br.set_handle_refresh(False)  # can sometimes hang without this
  br.addheaders = [('User-agent', 'Firefox')]

  url = "https://cogcc.state.co.us/production/ReportPortal"

  # open the web page
  response = br.open(url)

  # Submit the current year into the form, click the button
  br.form = list(br.forms())[0]
  control = br.form.find_control('ctl00$mainContent$tbxYear')
  control.value = str(year)
  control = br.form.find_control(id="mainContent_btnSubmit")
  response = br.submit()

  # Send it into pandas
  contents = response.read()
  dfs = pd.read_html(contents)

  # Get our dataframes from the tables, just the last (totals) row
  df_oil = pd.DataFrame(dfs[0][-1:])
  df_gas = pd.DataFrame(dfs[2][-1:])
  df_water = pd.DataFrame(dfs[8][-1:])

  # Make a Series, return it
  totals = pd.Series()
  totals['Oil (BBLs)'] = int(df_oil['Annual'])
  totals['Gas (MCFs)'] = int(df_gas['Annual'])
  totals['Produced Water (BBLs)'] = int(df_water['Annual'])

  return totals

In [ ]:
#@title Make a new dataframe, get production for this year and last, save csv
df = pd.DataFrame()
df[year] = getproduction(year)
df[lastyear] = getproduction(lastyear)
df['Increase/Decrease'] = df[year] - df[lastyear]

# Save it to a CSV
df.to_csv(f'{workingdir}/colorado/production_totals.csv')

df

,2024,2023,Increase/Decrease
Oil (BBLs),169613612,166742960,2870652
Gas (MCFs),1848524385,1826768120,21756265
Produced Water (BBLs),279745071,288821247,-9076176


In [ ]:
#@title Make a new dataframe, get production since 2000, save csv
df = pd.DataFrame()

#start with this year, go back to 2000
whichyear = year
#for i in range(10):
while whichyear >= 2000:
  print(f'Downloading {whichyear}...')
  df[whichyear] = getproduction(whichyear)
  whichyear = whichyear - 1

# Save it to a CSV
df.to_csv(f'{workingdir}/colorado/production_history.csv')

df


,2024,2023,2022,2021,2020,2019,2018,2017,2016,2015,...,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000
Oil (BBLs),169613612,166742960,160255936,153507645,171513011,192233530,169142474,129944667,115967716,122131621,...,30502545,30022975,26224277,24576045,23218840,22570224,21600938,20571427,20195733,20018969
Gas (MCFs),1848524385,1826768120,1832378128,1898352805,2003881514,1995661277,1842010559,1697397588,1685453413,1685195431,...,1603824309,1556084461,1372655672,1266364156,1156096049,1098936310,1038552807,952604207,850382757,802546184
Produced Water (BBLs),279745071,288821247,284926307,282130419,283991301,334921299,337648365,309555462,314308892,329762941,...,358951633,367853932,394325688,398612933,347064221,295529495,302771061,283063400,267009155,253010584


### Alternate data sources (don't use these unless COGCC says they're more accurate than the county data)

In [ ]:
# #@title Download this year's and last year's data

# # Source 1: Monthly production reports from https://cogcc.state.co.us/data2.html#/downloads

# # This year
# url = f'https://cogcc.state.co.us/documents/data/downloads/production/{year}_prod_reports.csv'
# filename1 = (f'{workingdir}/co_production_{year}.csv')
# print(f"Downloading {filename1}...")
# request.urlretrieve(url, filename1)

# # Last year
# url = f'https://cogcc.state.co.us/documents/data/downloads/production/{lastyear}_prod_reports.csv'
# filename2 = (f'{workingdir}/co_production_{lastyear}.csv')
# print(f"Downloading {filename2}...")
# request.urlretrieve(url, filename2)

# # Headers
# url = 'https://cogcc.state.co.us/documents/data/downloads/production/monthly_prodheaders.csv'
# headerfile = (f'{workingdir}/co_production_headers.csv')
# print(f"Downloading {headerfile}...")
# request.urlretrieve(url, headerfile)

# # For taking the first and last lines off a text string
# def toptail(string):
#   lines = str(string).split('\n')[1:-1]
#   clean = '\n'.join(lines)
#   return clean


In [ ]:
# # Initialize output file
# outfile = (f'{workingdir}/colorado/productiondata.txt')
# print(f'Creating output file {outfile}')
# with open (outfile, 'w') as file:
#   file.write(f'{hr}Colorado production summary\n{hr}')

In [ ]:
# # get them both into a dataframe
# # data dictionary is at https://cogcc.state.co.us/documents/data/downloads/production/production_record_data_dictionary.htm
# # header is at https://cogcc.state.co.us/documents/data/downloads/production/monthly_prodheaders.csv

# # read the header row from the header file
# with open(headerfile, 'r') as f:
#     header = f.readline().strip().split(',')

# # clean up the headers, stripping brackets from column names
# pattern = r'\[|\]'
# header = [re.sub(pattern, '', item) for item in header]

# filename1 = (f'{workingdir}/co_production_{year}.csv')
# filename2 = (f'{workingdir}/co_production_{lastyear}.csv')

# df1 = pd.read_csv(filename1, header=None, names=header)
# df2 = pd.read_csv(filename2, header=None, names=header)

# df = pd.concat([df1, df2])

In [ ]:
# # Reality check on the number of rows
# print(df1.shape)
# print(df2.shape)
# print(df.shape)

In [ ]:
# # just get reports for this year and last
# df = df[df['report_year'] >= lastyear]
# # make the pivot table
# pivot = df.pivot_table(index='name',
#                        columns='report_year',
#                        values=['oil_vol', 'gas_prod', 'flared'],
#                        aggfunc='sum')

# pivot.to_csv(f'{workingdir}/colorado/production_by_company.csv')

# # Just comparing
# gas_total = pivot['gas_prod'].sum()
# flared_total = pivot['flared'].sum()
# oil_total = pivot['oil_vol'].sum()

# dfyear = df1[df1['report_year'] == year]
# gas_sum = dfyear['gas_prod'].sum()
# flared_sum = dfyear['flared'].sum()
# oil_sum = dfyear['oil_vol'].sum()

# out = (f'{hr}Source 1: All production reports\nhttps://cogcc.state.co.us/documents/data/downloads/production/{year}_prod_reports.csv\n{hr}')
# out += (f'Oil produced total (bbl):\n{toptail(oil_total)}\n')
# out += (f'Gas produced total (mcf):\n{toptail(gas_total)}\n')
# out += (f'Gas flared total (mcf):\n{toptail(flared_total)}\n\n')

# print(out)
# with open (outfile, 'a') as file:
#   file.write(out)

In [ ]:
# #@title Source 2: production summary (mdb) from https://cogcc.state.co.us/data2.html#/downloads

# # Install the mdbtools package for working with Access databases.
# # If we do this for real, need to download the zip file directly and unzip the mdb file inside.
# !apt install mdbtools

# # where's our file -- for now, assuming we're uploading it by hand
# mdbfile = '/content/coproduction.mdb'

# # Run a shell command to extract a csv from the Access database.
# # The table we care about is 'Colorado Annual Production'
# print('Extracting table from mdb to csv...')
# !mdb-export './coproduction.mdb' 'Colorado Annual Production' > '{workingdir}/coproduction2022.csv'

# print('Reading csv...')
# df = pd.read_csv(f'{workingdir}/coproduction2022.csv')

In [ ]:
# # We don't need to pivot if we're not looking at companies. Just sum the columns.
# #pivot = df.pivot_table(index='name', values=['flared_vented', 'oil_prod', 'gas_prod'], aggfunc='sum')

# # Sum the columns we care about.
# gas_total = df['gas_prod'].sum()
# flared_total = df['flared_vented'].sum()
# oil_total = df['oil_prod'].sum()

# out = (f'{hr}Source 2: Annual production summaries\nhttps://cogcc.state.co.us/documents/data/downloads/production/CO%202022%20Annual%20Production%20Summary-xp.zip\n{hr}')
# out += (f'Oil produced total (bbl):\n{comma.format(oil_total)}\n')
# out += (f'Gas produced total (mcf):\n{comma.format(gas_total)}\n')
# out += (f'Gas flared (mcf):\n{comma.format(flared_total)}\n\n')

# print(out)
# with open (outfile, 'a') as file:
#   file.write(out)


In [ ]:
# #@title Source 3: Production by county, generated from https://cogcc.state.co.us/data4.html#/production

# # Uploaded manually
# filename1 = '/content/2022_Production_by_County.xlsx'

# df_oil = pd.read_excel(filename1, sheet_name='Oil Produced')
# oil_sum = df_oil['Annual'].sum()
# df_gas = pd.read_excel(filename1, sheet_name='NG & Coalbed Gas Produced')
# gas_sum = df_gas['Annual'].sum()
# df_water = pd.read_excel(filename1, sheet_name='Water Produced')
# water_sum = df_water['Annual'].sum()

# out = (f'{hr}Source 3: Production by county\nGenerated from https://cogcc.state.co.us/data4.html#/production\n{hr}')
# out += (f'Oil produced total (bbl):\n{comma.format(oil_sum)}\n')
# out += (f'Gas produced total (mcf):\n{comma.format(gas_sum)}\n')
# out += (f'Water produced total (bbl):\n{comma.format(water_sum)}\n\n')

# print(out)
# with open (outfile, 'a') as file:
#   file.write(out)

# # Using this number for now as our actual totals
# totals = pd.Series()
# totals['Oil (BBL)'] = oil_sum
# totals['Gas (MCF)'] = gas_sum
# totals['Produced Water (BBL)'] = water_sum

# # totals.to_csv(f'{workingdir}/colorado/production_totals.csv')

In [ ]:
# #@title Source 4: Spill analysis PDF, manual data entry from https://cogcc.state.co.us/documents/data/downloads/environmental/SpillAnalysisByYear.pdf

# oil_sum = 135602845
# water_sum = 235513147

# out = (f'{hr}Source 4: Spill analysis PDF\nManual data entry from https://cogcc.state.co.us/documents/data/downloads/environmental/SpillAnalysisByYear.pdf\n{hr}')
# out += (f'Oil produced total (bbl):\n{comma.format(oil_sum)}\n')
# out += (f'Water produced total (bbl):\n{comma.format(water_sum)}\n\n')

# print(out)
# with open (outfile, 'a') as file:
#   file.write(out)

## New Mexico

In [ ]:
#@title Import the data from the web
#@markdown Click `Show code` if you get a DNS error to see how to fix this.
url = 'https://wwwapps.emnrd.nm.gov/ocd/ocdpermitting/Reporting/Production/ProductionInjectionSummaryReport.aspx'

# If you get a DNS error, open the URL above, save the webpage to NMProduction.html,
# and upload it to the data folder. Uncomment the next line to use the local file
# instead of downloading it.

url = './data/NMProduction.html'
dfs = pd.read_html(url)

NameError: name 'pd' is not defined

In [ ]:
#@title Process the data, save to csv

# How many rows in the first dataframe? If it's more than 400, use that.
# This is a kludge because when you download the data manually, pandas reads it
# twice for some reason I can't figure out. -Aaron
if len(dfs[0]) > 400:
  df = dfs[0]
else:   # If the first dataframe isn't huge, then combine all the dataframes and use that
  df = pd.concat([df for df in dfs])

# First row is column names
df.columns = df.iloc[4]
df = df[5:]
df = df.reindex()

# Print column names
print('Columns:')
print(df.columns)

# Drop all columns except 'Year', 'Month', 'Total Gas', and 'Total Oil'
df = df[['Year', 'Total Gas', 'Total Oil']]

# Drop the total rows by coercing to numbers then dropping NaNs
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df.dropna(subset=['Year'])

# Treat the year column as an integer
df.loc[:, 'Year'] = df['Year'].astype(int)

# Print the number of rows for each year
#print('Number of rows for each year:')
#print(df['Year'].value_counts())

# Treat the total gas and total oil columns as floats
df = df.astype({'Total Gas': float, 'Total Oil': float})

# Group by year
# Go back 10 years
#df = df[(df['Year'] >= year-9) & (df['Year'] <= year)]

# Go back to 2000
df = df[(df['Year'] >= 2000) & (df['Year'] <= year)]

grouped = df.groupby(['Year']).sum()
grouped['Oil (Gallons)'] = grouped['Total Oil'] * 42

# Rename, drop columns, and transpose
grouped = grouped.rename(columns={'Total Gas': 'Gas (MCFs)'}).drop(columns='Total Oil')
grouped = grouped.transpose()

# Convert column names to strings with no decimal points, Drop commas from the column names
grouped.columns = grouped.columns.astype(str).str.replace(r'\.0', '', regex=True)
grouped.columns = grouped.columns.str.replace(',', '')

# Save to csv
grouped.to_csv(f'{workingdir}/newmexico/production_totals.csv')

grouped

Columns:
Index(['Year', 'Month', 'Gas SE Oil Wells', 'Gas SE Gas Wells',
       'Gas NW Oil Wells', 'Gas NW Gas Wells', 'Gas Colfax Gas Wells',
       'Total Gas', 'Coalseam Gas', 'Oil SE Oil Wells', 'Oil SE Gas Wells',
       'Oil NW Oil Wells', 'Oil NW Gas Wells', 'Total Oil'],
      dtype='object', name=4)


Year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
4,,,,,,,,,,,,,,,,,,,,,
Gas (MCFs),"1,680,967,091","1,684,399,610","1,627,541,503","1,597,696,997","1,611,045,842","1,591,794,517","1,588,883,315","1,536,079,135","1,468,877,608","1,399,809,801",...,"1,275,014,653","1,264,414,225","1,294,501,549","1,515,995,153","1,797,139,260","1,969,697,640","2,259,157,833","2,712,773,249","3,131,911,942","3,525,140,803"
Oil (Gallons),"2,890,213,410","2,905,658,532","2,837,341,080","2,796,769,206","2,709,695,604","2,560,417,398","2,496,992,316","2,485,507,878","2,526,525,372","2,569,496,160",...,"6,219,966,606","6,158,612,250","7,237,878,690","10,475,020,122","14,150,372,628","15,807,079,260","19,457,037,894","24,766,783,902","28,174,659,828","31,103,361,366"


## Wyoming

In [ ]:
#@title Import the tables

from bs4 import BeautifulSoup
import requests

# Use BeautifulSoup to get the header row because Excel tables in HTML are stupid
url = (f'http://pipeline.wyo.gov/StateProduction.cfm?nYear={year}')
try:
  print("Downloading headers...")
  page = requests.get(url)
  page.raise_for_status()  # This will raise an exception for bad status codes (4xx or 5xx)
  soup = BeautifulSoup(page.text, "html.parser")
  table = soup.find(class_="xlMain")
except requests.exceptions.RequestException as e:
  print(f"Error: {e}")
  # Handle the error appropriately, e.g., exit the script, retry, etc.
except:
  # catches exceptions not related to request. This will run if the status_code is 503 or any code other than 200, because an error occurs while it attempts to run the following lines
  print("An exception occurred.")

# The table has 18 columns, so get the first 18 cells in the HTML table
header = table.find_all("td")[:18]
# That's our header row
headerrow = [x.get_text() for x in header]

# Get a dataframe for this year and last
# Note that this doesn't use BeautifulSoup
# Pandas reads the URL directly and gets tables from it
print(f"Getting {year}...")
tables = pd.read_html(url, attrs={'class': 'xlMain'})
df1 = pd.DataFrame(tables[0])

print(f"Getting {lastyear}...")
url = (f'http://pipeline.wyo.gov/StateProduction.cfm?nYear={lastyear}')
tables = pd.read_html(url, attrs={'class': 'xlMain'})
df2 = pd.DataFrame(tables[0])

#concatenate them, add a header row
df = pd.concat([df1, df2], ignore_index=True)
df.columns = headerrow

Getting 2024...
Getting 2023...


In [ ]:
#@title Import the tables for the last 10 years

from bs4 import BeautifulSoup
import requests

# Use BeautifulSoup to get the header row because Excel tables in HTML are stupid
url = (f'http://pipeline.wyo.gov/StateProduction.cfm?nYear={year}')
try:
  print("Downloading headers...")
  page = requests.get(url)
  page.raise_for_status()  # This will raise an exception for bad status codes (4xx or 5xx)
  soup = BeautifulSoup(page.text, "html.parser")
  table = soup.find(class_="xlMain")
except requests.exceptions.RequestException as e:
  print(f"Error: {e}")
  # Handle the error appropriately, e.g., exit the script, retry, etc.
except:
  # catches exceptions not related to request. This will run if the status_code is 503 or any code other than 200, because an error occurs while it attempts to run the following lines
  print("An exception occurred.")

# The table has 18 columns, so get the first 18 cells in the HTML table
header = table.find_all("td")[:18]
# That's our header row
headerrow = [x.get_text() for x in header]

# Create an empty list to store DataFrames for each year
dfs = []

# Loop through the past 10 years, starting with the current year
for i in range(10):
    current_year = year - i  # Calculate the year for this iteration
    print(f"Getting {current_year}...")

    current_url = f'http://pipeline.wyo.gov/StateProduction.cfm?nYear={current_year}'

    # Read HTML tables from the URL
    tables = pd.read_html(current_url, attrs={'class': 'xlMain'})

    # Append the first table (tables[0]) to the list of DataFrames
    dfs.append(pd.DataFrame(tables[0]))

# Concatenate all DataFrames in the list into a single DataFrame
df = pd.concat(dfs, ignore_index=True)

# Add the header row to the combined DataFrame
df.columns = headerrow

# print the frame
df.head(10)

Getting 2024...
Getting 2023...
Getting 2022...
Getting 2021...
Getting 2020...
Getting 2019...
Getting 2018...
Getting 2017...
Getting 2016...
Getting 2015...


,Year,Month,TotalWells,ProducingWells,IdleWells,IdleOil,IdleGas,OilBbls,CondensateBbls,Total OilBbls,ProducingOil Wells,Oil Bbls/DayOil Only,GasMcf,Casing HeadGasMcf,Total GasMcf,ProducingGas Wells,Gas Mcf/DayGas Wells Only,Water Bbls
0,2024,1,37419,26430,10989,5245,5744,7938047,645437,8583484,9866,29,85505269,31221587,116726856,16539,172,134947326
1,2024,2,37373,26215,11158,5328,5830,8038886,567514,8606400,9765,30,80359040,30874900,111233940,16425,173,126746608
2,2024,3,37360,26148,11212,5356,5856,8573281,577927,9151208,9766,30,83994242,32804038,116798280,16358,171,135825143
3,2024,4,37371,26134,11237,5274,5963,8292928,551819,8844747,9870,30,77382175,32149190,109531365,16240,167,129528165
4,2024,5,37346,26067,11279,5325,5954,8381174,552789,8933963,9826,29,79714011,32965955,112679966,16219,165,130725263
5,2024,6,37366,26380,10986,5114,5872,8180684,497319,8678003,10060,29,79429436,31139062,110568498,16295,168,127539913
6,2024,7,37368,26605,10763,5096,5667,8496059,512789,9008848,10077,29,80171577,32974973,113146550,16507,162,131238197
7,2024,8,37378,26618,10760,5097,5663,8688085,518658,9206743,10101,29,78439236,32931182,111370418,16497,164,131985184
8,2024,9,37400,26535,10865,5211,5654,8027058,507051,8534109,10000,28,76657446,31633400,108290846,16512,166,128406781
9,2024,10,37278,26569,10709,5189,5520,8456601,540130,8996731,10025,29,71173122,32884532,104057654,16516,160,134656568


In [ ]:
# Get numbers for each year, save to CSV
pivot = df.pivot_table(index = 'Year',
                       values = ['OilBbls', 'GasMcf', 'Water Bbls'],
                       aggfunc='sum')
pivot = pivot.transpose()
pivot.loc['Oil (Gallons)'] = pivot.loc['OilBbls'] * 42
pivot[f'Increase/Decrease, {year}-{lastyear}'] = pivot[year] - pivot[lastyear]
pivot = pivot.rename(index={'OilBbls': 'Oil (BBLs)', 'GasMcf': 'Gas (MCFs)', 'Water Bbls': 'Water (BBLs)'})
pivot.to_csv(f'{workingdir}/wyoming/production_totals.csv')

# add commas to pivot
pivot = pivot.applymap(lambda x: '{:,.0f}'.format(x) if isinstance(x, (int, float)) else x)
pivot

Year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,"Increase/Decrease, 2024-2023"
Gas (MCFs),"1,757,072,529","1,631,380,998","1,572,570,712","1,533,419,586","1,329,088,921","1,196,129,367","1,122,886,722","1,041,967,304","966,682,921","956,395,370","-10,287,551"
Oil (BBLs),"74,382,258","61,132,104","64,425,667","76,727,805","92,019,958","81,328,875","78,859,448","85,046,800","90,169,244","100,078,627","9,909,383"
Water (BBLs),"1,914,265,702","1,759,959,819","1,708,103,815","1,692,052,602","1,688,521,637","1,393,788,398","1,520,954,128","1,668,762,785","1,629,062,312","1,572,942,387","-56,119,925"
Oil (Gallons),"3,124,054,836","2,567,548,368","2,705,878,014","3,222,567,810","3,864,838,236","3,415,812,750","3,312,096,816","3,571,965,600","3,787,108,248","4,203,302,334","416,194,086"


# Make combined dataframes for overview graphics
## In case we want to automate this (or update numbers going back 5 years) in the future

- ~~Dataframe 1: How much spilled in each state?~~
-- ~~Columns: Colorado, New Mexico, Wyoming, Total~~
-- ~~Rows: All Spills (count), Produced Water (gal), Oil (gal), Total Volume Spilled (gal)~~
## DONE!

- Dataframe 2: How have spills changed?
-- Rows: Colorado, New Mexico, Wyoming
-- Columns: Number of spills, year through year-4

- Dataframe 3: How has production changed?
-- Columns: Colorado, New Mexico, Wyoming
-- Rows: Oil, Methane
-- Sheets: One each for year through year-4

---

- In setup, initialize combined dataframes
-- Import previous years' numbers (from where?)
- In each state, add to combined dataframes
- After production section, export combined dataframes

In [ ]:
#@title How much spilled in each state this year?

# Get the total volume of oil and produced water spilled via /colorado/co allcounts {year}.csv, /newmexico/volumespilled.csv, and /wyoming/wy material_volume {year}.csv
# Import these 3 files into dataframes: '{workingdir}/wyoming/wy material_volume {year}.csv' '{workingdir}/newmexico/volumespilled.csv' '/colorado/co allcounts {year}.csv'

wyvolume = pd.read_csv(f'{workingdir}/wyoming/wy material_volume {year}.csv', index_col=0)
wycount = pd.read_csv(f'{workingdir}/wyoming/wy material_types {year}.csv', index_col=0)
nmvolume = pd.read_csv(f'{workingdir}/newmexico/volumespilled.csv', index_col=0)
nmcount = pd.read_csv(f'{workingdir}/newmexico/summary.csv', index_col=0)
covolume = pd.read_csv(f'{workingdir}/colorado/co allcounts {year}.csv', index_col=0)

# Create a dataframe called states_summary
states_summary = pd.DataFrame()
states_summary = pd.DataFrame(columns=['All Spills (count)', 'Produced Water (gal)', 'Oil (gal)', 'Total Volume Spilled (gal)'],
                              index=['Colorado', 'New Mexico', 'Wyoming'])

# Populate the dataframe with values
# Populate Colorado
states_summary.loc['Colorado'] = [covolume['Number of Spills']['Total'],  # All Spills (count)
                                  covolume['Total Spill Volume']['Produced Water'],  # Produced Water (gal)
                                  covolume['Total Spill Volume']['Oil'], # Oil spilled (gal)
                                  covolume['Total Spill Volume']['Total'] # Total Volume Spilled (gal)
                                  ]

# add up the gallons spilled in NM
nm_liquid_total = (
    nmvolume['Total']['Crude oil spilled (gallons)'] +
    nmvolume['Total']['Produced water spilled (gallons)'] +
    nmvolume['Total']['Condensate spilled (gallons)'] +
    nmvolume['Total']['Other liquid spilled (gallons)']
)

# Populate New Mexico
states_summary.loc['New Mexico'] = [
    nmcount['Total']['Number of liquid spills'],
    nmvolume['Total']['Produced water spilled (gallons)'],
    nmvolume['Total']['Crude oil spilled (gallons)'],
    nm_liquid_total
]

# Populate Wyoming
states_summary.loc['Wyoming'] = [
    wycount['Number of spills'].sum(),
    wyvolume['Volume spilled (gallons)']['Produced Water'],
    wyvolume['Volume spilled (gallons)']['Oil'],
    wyvolume['Volume spilled (gallons)'].sum()
]

# Add a total row
states_summary.loc['Total'] = states_summary.sum(axis=0)

# Save the dataframe to {workingdir}/states_summary.csv
states_summary.to_csv(f'{workingdir}/states_summary.csv')

# Display the dataframe
states_summary

,All Spills (count),Produced Water (gal),Oil (gal),Total Volume Spilled (gal)
Colorado,545,"245,700","64,386","436,128"
New Mexico,"1,344","3,655,176","520,758","4,296,709"
Wyoming,972,1524542,190520,1808996
Total,"2,861","5,425,418","775,664","6,541,833"


In [ ]:
#@title How has production changed over time?

# Colorado history in f'{workingdir}/colorado/production_history.csv'
# New Mexico history in f'{workingdir}/newmexico/production_totals.csv'
# Wyoming history in f'{workingdir}/wyoming/production_totals.csv'

# Sheets: Year through Year-4
# Columns: Colorado, New Mexico, Wyoming
# Rows: Oil, Methane



In [ ]:
#@title How have spills changed over time?

# Get historic data from past spills reports - we need to put those in one sheet
